# Armocromia Season Detection — Versi 3 (bebas OOM, hierarki multi-model)
**Pipeline:** `Pre-processing → Augmentasi (GPU, color-safe) → Pemodelan (CNN / U-Net / Patch-Attention / LightGBM) → Ensemble hierarkis → Hasil`

## Kenapa Versi 2 masih OOM
| Saran yang diterima | Penilaian |
|---|---|
| Simpan `loss.item()` / `preds.detach().cpu()` | Benar secara umum, tapi **Versi 2 sudah melakukannya** (`run += loss.item()`, prediksi `.cpu().numpy()`). Bukan penyebabnya. |
| Batasi `IMG_CACHE` dengan `lru_cache` / FIFO | **Tidak tepat.** `IMG_CACHE` adalah memmap `.npy` di disk, bukan dict Python. Halamannya masuk *page cache* OS yang bisa di-reclaim, jadi `lru_cache` tidak relevan. |
| `gc.collect()` / `empty_cache()` di akhir epoch | Versi 2 sudah memakai `release_ram()`. Cara ini tidak bisa mengembalikan fragmentasi heap glibc, konteks CUDA, dan cache cuDNN yang menumpuk saat 5 fold + 3 model berjalan di **satu proses**. |

**Penyebab yang lebih mungkin:**
1. **Semua fold & model hidup dalam satu proses kernel.** Setiap fold membuat model, `DataParallel`, optimizer, dan workspace cuDNN baru. Sisa RAM/VRAM menumpuk sampai kernel dibunuh.
2. **ConvNeXt-S @384 px, batch 32/GPU dengan `DataParallel`.** GPU0 menampung batch penuh (64), augmentasi GPU (float32 64×3×442²), dan *gather* output. Di T4 16 GB, ini sangat dekat dengan CUDA OOM.

## Perbaikan Versi 3
| Lapisan | Perbaikan |
|---|---|
| Proses | **1 subprocess = 1 model × 1 fold × 1 GPU.** Saat proses selesai, OS mengambil kembali **semua** RAM & VRAM. Kebocoran tidak bisa menumpuk. Dua GPU T4 menjalankan 2 job paralel (bukan `DataParallel`). |
| Notebook | Proses notebook tidak pernah memuat model deep atau konteks CUDA, jadi RAM-nya tetap kecil. |
| VRAM | Batch fisik di-*probe* otomatis (<80% VRAM) + gradient accumulation untuk batch efektif. Bila tetap OOM, batch **dibelah otomatis** (tidak crash). Ada AMP FP16, `channels_last`, gradient checkpointing (ConvNeXt), dan `expandable_segments`. |
| RAM | Cache uint8 memmap (crop bbox wajah + letterbox), loader tanpa worker multiprocessing, `MALLOC_ARENA_MAX=2`. Bobot terbaik ditulis langsung ke disk (tanpa salinan `state_dict` di RAM). |
| Resume | Setiap fold menyimpan `.npy` + `.json`. Bila sesi habis, attach output lewat *Add Input → Your Work*, lalu job yang sudah selesai dilewati. |

## Hierarki model
```
FINAL (dipilih otomatis berdasarkan OOF: soft-vote rata / soft-vote Optuna / stacking LogReg) [+ chromatic arbiter]
├─ [CNN]   resnet50 @320 · effnetv2_s @320 · convnext_t @288     (head season + sub-season)
├─ [SEG]   unet_r34 @320   U-Net multi-task: segmentasi organ + region pooling per organ → klasifikasi
├─ [PATCH] patch_attn      patch kulit / iris / rambut → Transformer antar-organ
└─ [COLOR] lgb_flat (12 kelas) · lgb_hier (season → expert sub-season per season)
          setiap model ← 5 fold (OOF untuk kalibrasi & tuning bobot, test = rata-rata fold)
```

## 0. Tulis modul (`%%writefile`) & dependensi

In [ ]:
%%writefile armo_common.py
# armo_common.py — konfigurasi, path, label, metrik hierarkis, model zoo & utilitas memori (dipakai prep, worker, driver)
import os, glob, gc, json, math, random, ctypes
import numpy as np

KAGGLE_DATA_DIR = "/kaggle/input/datasets/sonymikio/deep-armocromina/Deep Armocromina"
LOCAL_DATA_DIR = "/home/sony/Downloads/Deep Armocromina for code"
IS_KAGGLE = os.path.exists("/kaggle/working")
SMOKE = os.environ.get("ARMO_SMOKE", "0") == "1"      # subset kecil + backbone kecil (uji cepat di CPU)
SEED = 42


def find_base():
    if os.environ.get("ARMO_BASE"):
        return os.environ["ARMO_BASE"]
    if os.path.exists(os.path.join(KAGGLE_DATA_DIR, "release/annotations.csv")):
        return KAGGLE_DATA_DIR
    if os.path.exists("/kaggle/input"):
        hits = glob.glob("/kaggle/input/*/release/annotations.csv") + \
               glob.glob("/kaggle/input/**/release/annotations.csv", recursive=True)
        if hits:
            return os.path.dirname(os.path.dirname(hits[0]))
    return LOCAL_DATA_DIR


def _is_tmpfs(d):
    # file di tmpfs = RAM yang dihitung ke limit memori container -> jangan taruh cache besar di sana
    best, fs = "", ""
    for line in open("/proc/mounts"):
        _, mnt, typ = line.split()[:3]
        if os.path.realpath(d).startswith(mnt) and len(mnt) > len(best):
            best, fs = mnt, typ
    return fs == "tmpfs"


BASE_DIR = find_base()
OUTPUT_DIR = os.environ.get("ARMO_OUTPUT") or (
    os.path.join("/kaggle/working" if IS_KAGGLE else LOCAL_DATA_DIR, "output_v3" + ("_smoke" if SMOKE else "")))
CACHE_DIR = os.environ.get("ARMO_CACHE") or (
    "/tmp/armo_cache" if IS_KAGGLE and not _is_tmpfs("/tmp") else os.path.join(OUTPUT_DIR, "_cache"))

RELEASE_DIR = os.path.join(BASE_DIR, "release")
ANNOTATIONS_CSV = os.path.join(RELEASE_DIR, "annotations.csv")
ORIG_DIR = os.path.join(BASE_DIR, "original_images_facer_masks/ORIGINAL_RGB_NOT_PROCESSED")
FACER_MASKS_DIR = os.path.join(BASE_DIR, "original_images_facer_masks/facer_masks")
MASK_CACHE_DIR = os.path.join(OUTPUT_DIR, "part_masks")
PRED_DIR = os.path.join(OUTPUT_DIR, "preds")
WEIGHT_DIR = os.path.join(OUTPUT_DIR, "weights")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")
META_CSV = os.path.join(OUTPUT_DIR, "meta.csv")
FEAT_CACHE = os.path.join(OUTPUT_DIR, "color_features.parquet")
PATCH_CACHE = os.path.join(OUTPUT_DIR, "patches.npz")
for _d in [OUTPUT_DIR, CACHE_DIR, MASK_CACHE_DIR, PRED_DIR, WEIGHT_DIR, LOG_DIR]:
    os.makedirs(_d, exist_ok=True)

N_FOLDS = 2 if SMOKE else 5
CACHE_SIDE = 112 if SMOKE else 352   # sisi cache uint8 (>= img_size terbesar); di-crop ke bbox wajah lalu letterbox
PATCH = 64


def img_cache_paths(n):
    s = CACHE_SIDE
    return dict(rgbm=os.path.join(CACHE_DIR, f"rgbm_{s}_{n}.npy"),
                orig=os.path.join(CACHE_DIR, f"orig_{s}_{n}.npy"),
                lm=os.path.join(CACHE_DIR, f"lm_{s}_{n}.npy"))


# ---------------- label hierarkis ----------------
SEASONS = ["autunno", "estate", "inverno", "primavera"]
SUBCLASSES = [("autunno", "deep"), ("autunno", "soft"), ("autunno", "warm"),
              ("estate", "cool"), ("estate", "light"), ("estate", "soft"),
              ("inverno", "bright"), ("inverno", "cool"), ("inverno", "deep"),
              ("primavera", "bright"), ("primavera", "light"), ("primavera", "warm")]
SUB_NAMES = [f"{s}-{c}" for s, c in SUBCLASSES]
NUM_SEASONS, NUM_CLASSES = len(SEASONS), len(SUBCLASSES)
SUB2SEASON = np.array([SEASONS.index(s) for s, _ in SUBCLASSES])
WARM_SEASONS = {SEASONS.index("autunno"), SEASONS.index("primavera")}
PART_ID = {"bg": 0, "skin": 1, "hair": 2, "eye": 3, "brow": 4, "lip": 5, "nose": 6, "mouth": 7}
N_PARTS = 8

# ---------------- model zoo (hierarki: family -> model -> fold) ----------------
# img_size / batch = titik awal yang aman untuk 1x Tesla T4 16 GB + AMP FP16 (1 proses = 1 GPU, tanpa DataParallel).
# "batch" adalah batch EFEKTIF; worker otomatis memeriksa batch fisik yang muat di VRAM dan menutup sisanya
# dengan gradient accumulation, serta membelah batch otomatis bila tetap OOM di tengah training.
MODEL_ZOO = {
    "resnet50":   dict(family="CNN",   kind="cls",   backbone="resnet50.a1_in1k",                  img_size=320, batch=32, epochs=15, lr=2e-4, grad_ckpt=False),
    "effnetv2_s": dict(family="CNN",   kind="cls",   backbone="tf_efficientnetv2_s.in21k_ft_in1k", img_size=320, batch=32, epochs=15, lr=2e-4, grad_ckpt=False),
    "convnext_t": dict(family="CNN",   kind="cls",   backbone="convnext_tiny.fb_in22k_ft_in1k",    img_size=288, batch=32, epochs=15, lr=1e-4, grad_ckpt=True),
    "unet_r34":   dict(family="SEG",   kind="unet",  backbone="resnet34.a1_in1k",                  img_size=320, batch=24, epochs=15, lr=2e-4, grad_ckpt=False),
    "patch_attn": dict(family="PATCH", kind="patch", backbone="resnet18.a1_in1k",                  img_size=PATCH, batch=128, epochs=30, lr=3e-4, grad_ckpt=False),
}
if SMOKE:
    for _m in MODEL_ZOO.values():
        _m.update(backbone="resnet18.a1_in1k", epochs=1, grad_ckpt=False)
        if _m["kind"] != "patch":
            _m.update(img_size=96, batch=8)
        else:
            _m.update(batch=32)
COMMON = dict(wd=1e-2, warmup_epochs=1, label_smoothing=0.1, lambda_season=1.0, lambda_sub=1.0,
              lambda_consist=0.5, lambda_seg=0.5, patience=5, head_lr_mult=5.0)


def seed_everything(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError:
        pass


# ---------------- metrik hierarkis ----------------
def season_probs(p12):
    return np.stack([p12[:, SUB2SEASON == s].sum(1) for s in range(NUM_SEASONS)], 1)


def hierarchical_decode(p12):
    # P(season) = sum anak -> pilih season -> pilih sub terbaik di dalam season itu (konsisten 100%)
    ps = season_probs(p12)
    s_hat = ps.argmax(1)
    masked = np.where(SUB2SEASON[None, :] == s_hat[:, None], p12, -1)
    return s_hat, masked.argmax(1)


def hier_metrics(y_sub, p12, s_hat=None, sub_hat=None):
    from sklearn.metrics import accuracy_score, f1_score
    if sub_hat is None:
        s_hat, sub_hat = hierarchical_decode(p12)
    y_s = SUB2SEASON[y_sub]
    ok_s = s_hat == y_s
    return dict(season_acc=accuracy_score(y_s, s_hat),
                season_f1=f1_score(y_s, s_hat, average="macro", labels=list(range(NUM_SEASONS)), zero_division=0),
                sub_acc=accuracy_score(y_sub, sub_hat),
                sub_f1=f1_score(y_sub, sub_hat, average="macro", labels=list(range(NUM_CLASSES)), zero_division=0),
                cond_sub_acc=float((sub_hat == y_sub)[ok_s].mean()) if ok_s.any() else 0.0)


# ---------------- memori ----------------
def ram_gb():
    import psutil
    p, tot = psutil.Process(), 0
    for q in [p, *p.children(recursive=True)]:
        try:
            tot += q.memory_info().rss
        except psutil.Error:
            pass
    return tot / 1e9


def cgroup_gb():
    # angka yang dipakai OOM-killer Kaggle (termasuk page cache & /dev/shm)
    for f in ["/sys/fs/cgroup/memory.current", "/sys/fs/cgroup/memory/memory.usage_in_bytes"]:
        if os.path.exists(f):
            return int(open(f).read()) / 1e9
    return float("nan")


def release_ram():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)   # kembalikan heap glibc yang terfragmentasi ke OS
    except OSError:
        pass

In [ ]:
%%writefile armo_prep.py
# armo_prep.py — TAHAP 1: PRE-PROCESSING (dijalankan sebagai subprocess terpisah -> RAM/VRAM pyfacer lepas total saat selesai)
#   1) anotasi + fold   2) masker organ (facer asli -> pyfacer FaRL -> heuristik)   3) fitur warna RGB/HSV/Lab + patch organ
#   4) cache citra uint8 (memmap .npy, crop bbox wajah + letterbox) untuk RGB-M dan ORIGINAL+labelmap
import os, sys, glob, math, subprocess
import numpy as np, pandas as pd, cv2
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from armo_common import *

cv2.setNumThreads(1)
N_THREADS = min(8, os.cpu_count() or 4)


def log(*a):
    print("[prep]", *a, flush=True)


# ---------- 1. anotasi ----------
def load_df():
    df = pd.read_csv(ANNOTATIONS_CSV)
    df["key"] = df["path_rgb_original"].str.replace(r"^RGB/", "", regex=True).str.replace(r"\.[^.]+$", "", regex=True)
    for c in ["path_rgb_original", "path_rgb_masked", "path_mask"]:   # 6 path punya spasi sebelum ekstensi
        df[c] = df[c].str.replace(r"\s+(\.[^.]+)$", r"\1", regex=True)
    df["y_sub"] = [SUB_NAMES.index(f"{c}-{s}") for c, s in zip(df["class"], df["sub_class"])]
    df["y_season"] = SUB2SEASON[df["y_sub"]]
    orig = {os.path.splitext(os.path.relpath(p, ORIG_DIR))[0].rstrip(): p for p in glob.glob(os.path.join(ORIG_DIR, "*/*/*/*"))}
    df["orig_path"] = df["key"].str.rstrip().map(orig)
    assert df["orig_path"].notna().all(), df.loc[df["orig_path"].isna(), "key"].tolist()[:10]
    df["rgbm_path"] = [os.path.join(RELEASE_DIR, p) for p in df["path_rgb_masked"]]
    df["has_facer"] = [os.path.exists(os.path.join(FACER_MASKS_DIR, k + "_face.png")) for k in df["key"]]
    if SMOKE:
        df = df.sample(frac=1.0, random_state=0).groupby(["partition", "y_sub"]).head(12)
    df = df.reset_index(drop=True)
    from sklearn.model_selection import StratifiedKFold
    df["fold"] = -1
    tr = np.where(df.partition.values == "train")[0]
    for k, (_, va) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(tr, df.y_sub.values[tr])):
        df.loc[tr[va], "fold"] = k
    return df


# ---------- 2. masker organ ----------
FACER_FILE2ID = [("hair", 2), ("face", 1), ("nose", 6), ("l_brow", 4), ("r_brow", 4), ("l_eye", 3), ("r_eye", 3),
                 ("u_lip", 5), ("l_lip", 5), ("mouth", 7)]
LAPA2ID = {"background": 0, "face": 1, "rb": 4, "lb": 4, "re": 3, "le": 3, "nose": 6, "ulip": 5, "imouth": 7, "llip": 5, "hair": 2}


def mask_cache_path(key):
    return os.path.join(MASK_CACHE_DIR, key.replace("/", "__") + ".png")


def labelmap_from_facer_files(key):
    lm = None
    for name, pid in FACER_FILE2ID:
        m = np.array(Image.open(os.path.join(FACER_MASKS_DIR, f"{key}_{name}.png")).convert("L")) > 127
        if lm is None:
            lm = np.zeros(m.shape, np.uint8)
        lm[m] = pid
    return lm


def heuristic_labelmap(rgb):
    fg = rgb.max(-1) > 12
    ycc = cv2.cvtColor(rgb, cv2.COLOR_RGB2YCrCb)
    skin = fg & (ycc[..., 1] >= 135) & (ycc[..., 1] <= 180) & (ycc[..., 2] >= 85) & (ycc[..., 2] <= 135)
    skin = cv2.morphologyEx(skin.astype(np.uint8), cv2.MORPH_OPEN, np.ones((5, 5), np.uint8)).astype(bool)
    lm = np.zeros(rgb.shape[:2], np.uint8)
    lm[skin] = 1
    if skin.any():
        ys = np.where(skin)[0]
        hair = fg & ~skin
        hair[int(np.percentile(ys, 60)):] = False
        lm[hair] = 2
    return lm


def load_facer():
    if os.environ.get("ARMO_NO_FACER") == "1":
        return None
    try:
        import facer
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyfacer", "validators"], check=False)
        try:
            import facer
        except Exception as e:
            log("pyfacer tidak tersedia -> heuristik:", e)
            return None
    return facer


def build_masks(df):
    src_csv = os.path.join(MASK_CACHE_DIR, "_sources.csv")
    known = dict(pd.read_csv(src_csv, keep_default_na=False).values) if os.path.exists(src_csv) else {}
    todo = [r for r in df.itertuples() if not os.path.exists(mask_cache_path(r.key))]
    log(f"masker: {len(df) - len(todo)} dari cache, {len(todo)} dibuat")
    need_facer = any(not r.has_facer for r in todo)
    facer = load_facer() if need_facer else None
    if facer is not None:
        import torch
        dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        det = facer.face_detector("retinaface/mobilenet", device=dev)
        parse = facer.face_parser("farl/lapa/448", device=dev)
    for i, r in enumerate(todo):
        lm, s = None, None
        if r.has_facer:
            lm, s = labelmap_from_facer_files(r.key), "facer_files"
        elif facer is not None:
            try:
                rgb = np.array(Image.open(r.orig_path).convert("RGB"))
                img = facer.hwc2bchw(torch.from_numpy(rgb)).to(dev)
                with torch.inference_mode():
                    faces = det(img)
                    if len(faces.get("scores", [])) > 0:
                        b = int(torch.argmax(faces["scores"]))
                        faces = parse(img, {k: v[b:b + 1] for k, v in faces.items()})
                        seg = faces["seg"]["logits"].argmax(1)[0].cpu().numpy()
                        lm, s = np.array([LAPA2ID[n] for n in faces["seg"]["label_names"]], np.uint8)[seg], "pyfacer"
                del img, faces
            except Exception as e:
                log("pyfacer error", r.key, e)
        if lm is None:
            lm, s = heuristic_labelmap(np.array(Image.open(r.orig_path).convert("RGB"))), "heuristic"
        Image.fromarray(lm).save(mask_cache_path(r.key))
        known[r.key] = s
        if (i + 1) % 500 == 0:
            log(f"  masker {i + 1}/{len(todo)} | RAM {ram_gb():.1f} GB")
            pd.DataFrame(list(known.items()), columns=["key", "src"]).to_csv(src_csv, index=False)
    pd.DataFrame(list(known.items()), columns=["key", "src"]).to_csv(src_csv, index=False)
    return [known.get(k, "facer_files" if h else "unknown") for k, h in zip(df["key"], df["has_facer"])]


def load_rgb_lm(orig_path, key):
    rgb = np.array(Image.open(orig_path).convert("RGB"))
    lm = np.array(Image.open(mask_cache_path(key)))
    if lm.shape != rgb.shape[:2]:
        lm = cv2.resize(lm, (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_NEAREST)
    return rgb, lm


# ---------- 3. fitur kolorimetri + patch organ ----------
REGIONS = ["skin", "hair", "eye", "lip", "brow"]


def srgb_to_lab(rgb_u8):
    lab = cv2.cvtColor(rgb_u8.reshape(-1, 1, 3), cv2.COLOR_RGB2LAB).reshape(-1, 3).astype(np.float32)
    lab[:, 0] *= 100.0 / 255.0; lab[:, 1:] -= 128.0
    return lab


def region_pixels(rgb, lm, region):
    m = lm == PART_ID[region]
    if region == "skin":
        m = cv2.erode(m.astype(np.uint8), np.ones((3, 3), np.uint8)).astype(bool)
    px = rgb[m]
    if len(px) < 20:
        return None, None
    lab = srgb_to_lab(px); L = lab[:, 0]
    if region == "skin":
        keep = (L >= 20) & (L <= 92)
        if keep.sum() >= 20:
            lo, hi = np.percentile(L[keep], [5, 95]); keep &= (L >= lo) & (L <= hi)
    elif region == "hair":
        keep = L <= np.percentile(L, 95)
    elif region == "eye":
        keep = L <= np.percentile(L, 60)          # buang sklera
    else:
        keep = np.ones(len(L), bool)
    return px[keep], lab[keep]


def region_features(px, lab, pre, total):
    if px is None:
        return {}
    p = px.astype(np.float32); s = p.sum(1) + 1e-6
    hsv = cv2.cvtColor(px.reshape(-1, 1, 3), cv2.COLOR_RGB2HSV_FULL).reshape(-1, 3).astype(np.float32)
    hue = hsv[:, 0] / 255.0 * 2 * np.pi
    a, b = lab[:, 1].mean(), lab[:, 2].mean()
    return {f"{pre}_R": p[:, 0].mean(), f"{pre}_G": p[:, 1].mean(), f"{pre}_B": p[:, 2].mean(),
            f"{pre}_Rstd": p[:, 0].std(), f"{pre}_Gstd": p[:, 1].std(), f"{pre}_Bstd": p[:, 2].std(),
            f"{pre}_rratio": (p[:, 0] / s).mean(), f"{pre}_bratio": (p[:, 2] / s).mean(),
            f"{pre}_H": (np.degrees(np.arctan2(np.sin(hue).mean(), np.cos(hue).mean())) + 360) % 360,
            f"{pre}_S": hsv[:, 1].mean() / 255.0, f"{pre}_V": hsv[:, 2].mean() / 255.0, f"{pre}_Sstd": hsv[:, 1].std() / 255.0,
            f"{pre}_L": lab[:, 0].mean(), f"{pre}_a": a, f"{pre}_b": b, f"{pre}_Lstd": lab[:, 0].std(),
            f"{pre}_bstd": lab[:, 2].std(), f"{pre}_C": float(np.hypot(a, b)),
            f"{pre}_hab": float((np.degrees(np.arctan2(b, a)) + 360) % 360), f"{pre}_frac": len(px) / max(total, 1)}


def extract_color_features(rgb, lm):
    fg = lm > 0; total = int(fg.sum()); f = {}
    for reg in REGIONS:
        f.update(region_features(*region_pixels(rgb, lm, reg), reg, total))
    px = rgb[fg].astype(np.float32) / 255.0
    if len(px):
        e = np.power(np.power(px, 6).mean(0), 1 / 6); e = e / (e.sum() + 1e-6)   # illuminant Shades-of-Gray
        f.update(illum_r=e[0], illum_g=e[1], illum_b=e[2])
    g = lambda k: f.get(k, np.nan)
    f["ITA"] = np.degrees(np.arctan2(g("skin_L") - 50, g("skin_b")))
    for reg in ["hair", "eye", "lip", "brow"]:
        f[f"dL_skin_{reg}"] = abs(g("skin_L") - g(f"{reg}_L"))
        f[f"db_skin_{reg}"] = g("skin_b") - g(f"{reg}_b")
        f[f"dE_skin_{reg}"] = np.sqrt((g("skin_L") - g(f"{reg}_L")) ** 2 + (g("skin_a") - g(f"{reg}_a")) ** 2 + (g("skin_b") - g(f"{reg}_b")) ** 2)
    f["michelson_skin_hair"] = (g("skin_L") - g("hair_L")) / (g("skin_L") + g("hair_L") + 1e-6)
    f["michelson_skin_eye"] = (g("skin_L") - g("eye_L")) / (g("skin_L") + g("eye_L") + 1e-6)
    c = [f["dL_skin_hair"], f["dL_skin_eye"], f["dL_skin_brow"]]
    f["contrast_max"] = np.nanmax(c) if np.isfinite(c).any() else np.nan
    cm = [g("skin_C"), g("hair_C"), g("eye_C")]
    f["chroma_mean"] = np.nanmean(cm) if np.isfinite(cm).any() else np.nan
    return f


def densest_window_crop(rgb, m, side):
    side = int(max(8, side))
    if m.sum() < 10:
        return None
    dens = cv2.blur(m.astype(np.float32), (side, side))
    y, x = np.unravel_index(np.argmax(dens), dens.shape)
    y0, x0 = max(0, y - side // 2), max(0, x - side // 2)
    crop = rgb[y0:y0 + side, x0:x0 + side] * m[y0:y0 + side, x0:x0 + side, None]
    return cv2.resize(crop.astype(np.uint8), (PATCH, PATCH), interpolation=cv2.INTER_AREA)


def eye_crop(rgb, lm):
    n, _, stats, _ = cv2.connectedComponentsWithStats((lm == PART_ID["eye"]).astype(np.uint8))
    if n <= 1:
        return None
    i = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    x, y, w, h = stats[i, :4]
    s = int(max(w, h) * 1.3) + 2
    y0, x0 = max(0, y + h // 2 - s // 2), max(0, x + w // 2 - s // 2)
    return cv2.resize(rgb[y0:y0 + s, x0:x0 + s], (PATCH, PATCH), interpolation=cv2.INTER_CUBIC)


def extract_patches(rgb, lm):
    side = math.sqrt(max((lm == PART_ID["skin"]).sum(), 1))
    out, ok = np.zeros((3, PATCH, PATCH, 3), np.uint8), np.zeros(3, np.float32)
    for i, p in enumerate([densest_window_crop(rgb, lm == PART_ID["skin"], 0.30 * side), eye_crop(rgb, lm),
                           densest_window_crop(rgb, lm == PART_ID["hair"], 0.30 * side)]):
        if p is not None:
            out[i], ok[i] = p, 1
    return out, ok


def build_features(df):
    if os.path.exists(FEAT_CACHE) and os.path.exists(PATCH_CACHE):
        feat = pd.read_parquet(FEAT_CACHE)
        if len(feat) == len(df) and (feat["key"].values == df["key"].values).all():
            log("fitur warna & patch dari cache")
            return
        log("cache fitur tidak cocok dengan meta -> dihitung ulang")
    n = len(df)
    rows = [None] * n
    patches, ok = np.zeros((n, 3, PATCH, PATCH, 3), np.uint8), np.zeros((n, 3), np.float32)

    def one(i):
        rgb, lm = load_rgb_lm(df.orig_path.values[i], df.key.values[i])
        rows[i] = extract_color_features(rgb, lm)
        patches[i], ok[i] = extract_patches(rgb, lm)

    with ThreadPoolExecutor(N_THREADS) as ex:
        for j, _ in enumerate(ex.map(one, range(n))):
            if (j + 1) % 1000 == 0:
                log(f"  fitur {j + 1}/{n} | RAM {ram_gb():.1f} GB")
    feat = pd.DataFrame(rows).astype(np.float32)
    feat.insert(0, "key", df["key"].values)
    feat.to_parquet(FEAT_CACHE)
    np.savez_compressed(PATCH_CACHE, patches=patches, ok=ok)
    log("fitur:", feat.shape[1] - 1, "kolom")


# ---------- 4. cache citra (memmap) ----------
def crop_bbox(fg, h, w, margin=0.08):
    ys, xs = np.where(fg)
    if len(ys) < 50:
        return 0, h, 0, w
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    my, mx = int((y1 - y0) * margin), int((x1 - x0) * margin)
    return max(0, y0 - my), min(h, y1 + my), max(0, x0 - mx), min(w, x1 + mx)


def letterbox(img, side, interp):
    h, w = img.shape[:2]; s = side / max(h, w)
    nh, nw = max(1, round(h * s)), max(1, round(w * s))
    r = cv2.resize(img, (nw, nh), interpolation=interp)
    out = np.zeros((side, side) + img.shape[2:], img.dtype)
    y0, x0 = (side - nh) // 2, (side - nw) // 2
    out[y0:y0 + nh, x0:x0 + nw] = r
    return out


def build_image_caches(df):
    paths, n, S = img_cache_paths(len(df)), len(df), CACHE_SIDE
    if all(os.path.exists(p) for p in paths.values()):
        log("cache citra sudah ada:", paths["rgbm"]); return
    rgbm = np.lib.format.open_memmap(paths["rgbm"] + ".tmp", mode="w+", dtype=np.uint8, shape=(n, S, S, 3))
    orig = np.lib.format.open_memmap(paths["orig"] + ".tmp", mode="w+", dtype=np.uint8, shape=(n, S, S, 3))
    lms = np.lib.format.open_memmap(paths["lm"] + ".tmp", mode="w+", dtype=np.uint8, shape=(n, S, S))

    def one(i):
        with Image.open(df.rgbm_path.values[i]) as im:
            a = np.array(im.convert("RGB"))
        y0, y1, x0, x1 = crop_bbox(a.max(-1) > 12, *a.shape[:2])        # buang latar hitam -> lebih banyak piksel wajah
        rgbm[i] = letterbox(a[y0:y1, x0:x1], S, cv2.INTER_AREA)
        rgb, lm = load_rgb_lm(df.orig_path.values[i], df.key.values[i])
        y0, y1, x0, x1 = crop_bbox(lm > 0, *lm.shape, margin=0.12)
        orig[i] = letterbox(rgb[y0:y1, x0:x1], S, cv2.INTER_AREA)
        lms[i] = letterbox(lm[y0:y1, x0:x1], S, cv2.INTER_NEAREST)

    with ThreadPoolExecutor(N_THREADS) as ex:
        for j, _ in enumerate(ex.map(one, range(n))):
            if (j + 1) % 1000 == 0:
                log(f"  cache citra {j + 1}/{n} | RAM {ram_gb():.1f} GB | cgroup {cgroup_gb():.1f} GB")
    for a in (rgbm, orig, lms):
        a.flush()
    del rgbm, orig, lms
    for p in paths.values():
        os.replace(p + ".tmp", p)
    log("cache citra:", {k: f"{os.path.getsize(p) / 1e9:.2f} GB" for k, p in paths.items()})


if __name__ == "__main__":
    log(f"BASE_DIR={BASE_DIR}\n       OUTPUT_DIR={OUTPUT_DIR} | CACHE_DIR={CACHE_DIR} | SMOKE={SMOKE}")
    df = load_df()
    log("total", len(df), "| train", (df.partition == "train").sum(), "| test", (df.partition == "test").sum())
    df["mask_src"] = build_masks(df)
    log("sumber masker:", df["mask_src"].value_counts().to_dict())
    build_features(df)
    build_image_caches(df)
    df[["key", "partition", "celeba", "y_sub", "y_season", "fold", "mask_src", "orig_path", "rgbm_path"]].to_csv(META_CSV, index=False)
    log(f"selesai | RAM {ram_gb():.1f} GB")

In [ ]:
%%writefile armo_train.py
# armo_train.py — TAHAP 2+3: AUGMENTASI (di GPU) + PEMODELAN untuk SATU model x SATU fold di SATU GPU.
# Dijalankan driver sebagai subprocess: ketika fold selesai proses mati -> seluruh RAM, VRAM, konteks CUDA,
# fragmentasi heap glibc & cache cuDNN dikembalikan ke OS. Kebocoran tidak bisa menumpuk lintas fold/model.
import os, sys, json, math, time, argparse, gc
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import timm
from armo_common import *

ap = argparse.ArgumentParser()
ap.add_argument("--model", required=True, choices=list(MODEL_ZOO))
ap.add_argument("--fold", type=int, required=True)
ap.add_argument("--img_size", type=int, default=0)
ap.add_argument("--epochs", type=int, default=0)
ap.add_argument("--batch", type=int, default=0)
ap.add_argument("--tag", default="")
ap.add_argument("--force", action="store_true")
args = ap.parse_args()

CFG = {**COMMON, **MODEL_ZOO[args.model]}
for k in ["img_size", "epochs", "batch"]:
    if getattr(args, k):
        CFG[k] = getattr(args, k)
KIND, K, TAG = CFG["kind"], args.fold, args.tag or args.model
OUT = {x: os.path.join(PRED_DIR, f"{TAG}_{x}_f{K}.npy") for x in ["oof", "te"]}
OUT_JSON = os.path.join(PRED_DIR, f"{TAG}_f{K}.json")
W_PATH = os.path.join(WEIGHT_DIR, f"{TAG}_f{K}.pt")
if all(os.path.exists(p) for p in [*OUT.values(), OUT_JSON]) and not args.force:
    print(f"[{TAG} f{K}] sudah ada, dilewati"); sys.exit(0)

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEV.type == "cuda"
seed_everything(SEED + K)
torch.backends.cudnn.benchmark = True


def log(*a):
    print(f"[{TAG} f{K}]", *a, flush=True)


meta = pd.read_csv(META_CSV, keep_default_na=False)
N = len(meta)
Y_SUB, Y_S = meta.y_sub.values.astype(np.int64), meta.y_season.values.astype(np.int64)
SEG_VALID = (~meta.mask_src.isin(["heuristic"])).values.astype(np.float32)   # masker heuristik tidak dipakai sbg target segmentasi
train_rows = np.where(meta.partition.values == "train")[0]
test_rows = np.where(meta.partition.values == "test")[0]
tr_rows = train_rows[meta.fold.values[train_rows] != K]
va_rows = train_rows[meta.fold.values[train_rows] == K]
CACHE = img_cache_paths(N)

# ======================= AUGMENTASI COLOR-SAFE (GPU) =======================
# flip, rotasi ±8°, crop 0.88–1, translasi 4%, Bradford ±250 K, kontras ±5%, blur, noise. TANPA hue shift / ColorJitter.
_M_BFD = np.array([[0.8951, 0.2664, -0.1614], [-0.7502, 1.7135, 0.0367], [0.0389, -0.0685, 1.0296]])
_M_RGB2XYZ = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])


def cct_to_xyz_white(T):
    x = (-4.6070e9 / T**3 + 2.9678e6 / T**2 + 0.09911e3 / T + 0.244063) if T <= 7000 else \
        (-2.0064e9 / T**3 + 1.9018e6 / T**2 + 0.24748e3 / T + 0.237040)
    y = -3.0 * x * x + 2.870 * x - 0.275
    return np.array([x / y, 1.0, (1 - x - y) / y])


_W_D65 = cct_to_xyz_white(6504)


def bradford_matrix(T):
    src, dst = _M_BFD @ _W_D65, _M_BFD @ cct_to_xyz_white(T)
    A = np.linalg.inv(_M_BFD) @ np.diag(dst / src) @ _M_BFD
    return np.linalg.inv(_M_RGB2XYZ) @ A @ _M_RGB2XYZ


def _gauss_blur(img, sigma):
    rad = max(1, int(round(4 * sigma)))
    k = torch.arange(-rad, rad + 1, device=img.device, dtype=img.dtype)
    k = torch.exp(-k ** 2 / (2 * sigma ** 2)); k = k / k.sum()
    img = F.pad(img[None], (rad, rad, rad, rad), mode="reflect")
    img = F.conv2d(img, k.view(1, 1, 1, -1).expand(3, 1, 1, -1).contiguous(), groups=3)
    img = F.conv2d(img, k.view(1, 1, -1, 1).expand(3, 1, -1, 1).contiguous(), groups=3)
    return img[0]


def gpu_aug(x_u8, m_u8, size, train):
    """x_u8: BxHxWx3 uint8 (GPU), m_u8: BxHxW labelmap atau None -> (Bx3xSxS float [0,1], BxSxS long)"""
    x = x_u8.permute(0, 3, 1, 2).float().div_(255.0)
    m = None if m_u8 is None else m_u8[:, None].float()
    if x.shape[-1] > size * 1.25:            # turunkan resolusi dulu (anti-aliasing) bila cache jauh lebih besar
        s0 = int(size * 1.1)
        x = F.interpolate(x, size=(s0, s0), mode="area")
        m = None if m is None else F.interpolate(m, size=(s0, s0), mode="nearest")
    if not train:
        x = F.interpolate(x, size=(size, size), mode="area")
        m = None if m is None else F.interpolate(m, size=(size, size), mode="nearest")[:, 0].long()
        return x, m
    B, dev = x.shape[0], x.device
    r = torch.empty(B, device=dev).uniform_(0.88, 1.0).sqrt()
    c = (torch.rand(B, 2, device=dev) * 2 - 1) * (1 - r)[:, None]
    flip = torch.where(torch.rand(B, device=dev) < 0.5, -1.0, 1.0)
    ang = torch.empty(B, device=dev).uniform_(-8, 8).deg2rad()
    t = torch.empty(B, 2, device=dev).uniform_(-0.04, 0.04) * 2
    cos, sin = ang.cos(), ang.sin()
    A = torch.stack([torch.stack([cos, sin], 1), torch.stack([-sin, cos], 1)], 1) * r[:, None, None]
    A[:, 0] *= flip[:, None]
    b = c - (A @ t[..., None])[..., 0]
    grid = F.affine_grid(torch.cat([A, b[..., None]], 2), (B, 3, size, size), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    if m is not None:
        m = F.grid_sample(m, grid, mode="nearest", padding_mode="zeros", align_corners=False)[:, 0].long()
    sel = torch.rand(B) < 0.5                                           # Bradford ±250 K di linear sRGB
    if sel.any():
        M = torch.tensor(np.stack([bradford_matrix(6504 + np.random.uniform(-250, 250)) for _ in range(int(sel.sum()))]),
                         dtype=torch.float32, device=dev)
        sd = sel.to(dev); xs = x[sd]
        lin = torch.where(xs <= 0.04045, xs / 12.92, ((xs + 0.055) / 1.055) ** 2.4)
        lin = torch.einsum("bij,bjhw->bihw", M, lin).clamp(0, 1)
        x[sd] = torch.where(lin <= 0.0031308, lin * 12.92, 1.055 * lin.clamp_min(1e-12) ** (1 / 2.4) - 0.055)
    sel = (torch.rand(B, device=dev) < 0.3)[:, None, None, None]        # kontras ±5%
    mu = x.mean((1, 2, 3), keepdim=True)
    x = torch.where(sel, (x - mu) * torch.empty(B, 1, 1, 1, device=dev).uniform_(0.95, 1.05) + mu, x)
    for j in torch.nonzero(torch.rand(B) < 0.2).flatten().tolist():    # blur
        x[j] = _gauss_blur(x[j], float(np.random.uniform(0.1, 0.8)))
    sel = (torch.rand(B, device=dev) < 0.2)[:, None, None, None]        # noise ISO
    x = torch.where(sel, x + torch.randn_like(x) * 0.01, x)
    return x.clamp_(0, 1), m


# ======================= LOADER (tanpa worker multiprocessing) =======================
class MemmapLoader:
    """Baca batch uint8 dari memmap .npy (page cache OS, bisa di-reclaim) dengan 1 thread prefetch, augmentasi di GPU.
    Tidak ada proses worker DataLoader / shared memory / dict cache di RAM -> RSS konstan antar epoch."""
    def __init__(self, rows, train, bs, size, mean, std):
        self.rows, self.train, self.bs, self.size = np.asarray(rows, np.int64), train, bs, size
        self.mean = torch.tensor(mean, dtype=torch.float32, device=DEV).view(1, 3, 1, 1)
        self.std = torch.tensor(std, dtype=torch.float32, device=DEV).view(1, 3, 1, 1)
        self.srcs = [CACHE["rgbm"]] if KIND == "cls" else [CACHE["orig"], CACHE["lm"]]

    def __len__(self):
        n = len(self.rows)
        return max(1, n // self.bs) if self.train else -(-n // self.bs)

    def __iter__(self):
        arrs = [np.load(p, mmap_mode="r") for p in self.srcs]
        order = np.random.permutation(len(self.rows)) if self.train else np.arange(len(self.rows))
        batches = [np.sort(order[i * self.bs:(i + 1) * self.bs]) for i in range(len(self))]
        fetch = lambda ix: [np.ascontiguousarray(a[self.rows[ix]]) for a in arrs]
        with ThreadPoolExecutor(1) as ex:
            fut = ex.submit(fetch, batches[0])
            for j, ix in enumerate(batches):
                data = fut.result()
                if j + 1 < len(batches):
                    fut = ex.submit(fetch, batches[j + 1])
                rr = self.rows[ix]
                with torch.no_grad():
                    x, m = gpu_aug(torch.from_numpy(data[0]).to(DEV), torch.from_numpy(data[1]).to(DEV) if len(data) > 1 else None,
                                   self.size, self.train)
                    x = ((x - self.mean) / self.std).contiguous(memory_format=torch.channels_last)
                yield dict(x=x, m=m, sv=torch.from_numpy(SEG_VALID[rr]).to(DEV),
                           ysub=torch.from_numpy(Y_SUB[rr]).to(DEV), ys=torch.from_numpy(Y_S[rr]).to(DEV))
                del data
        del arrs


class PatchLoader:
    _P = None

    def __init__(self, rows, train, bs, *_):
        if PatchLoader._P is None:
            z = np.load(PATCH_CACHE); PatchLoader._P = (z["patches"], z["ok"])
        self.rows, self.train, self.bs = np.asarray(rows, np.int64), train, bs
        self.mean = torch.tensor([0.485, 0.456, 0.406], device=DEV).view(1, 1, 3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225], device=DEV).view(1, 1, 3, 1, 1)

    def __len__(self):
        n = len(self.rows)
        return max(1, n // self.bs) if self.train else -(-n // self.bs)

    def __iter__(self):
        P, OK = PatchLoader._P
        order = np.random.permutation(len(self.rows)) if self.train else np.arange(len(self.rows))
        for i in range(len(self)):
            rr = self.rows[np.sort(order[i * self.bs:(i + 1) * self.bs])]
            with torch.no_grad():
                x = torch.from_numpy(P[rr]).to(DEV).float().div_(255).permute(0, 1, 4, 2, 3)   # B,3reg,C,H,W
                if self.train:
                    f = torch.rand(x.shape[0], device=DEV) < 0.5
                    x[f] = torch.flip(x[f], dims=[-1])
                    x = (x + 0.01 * torch.randn_like(x)).clamp(0, 1)
                x = (x - self.mean) / self.std
            yield dict(x=x, ok=torch.from_numpy(OK[rr]).to(DEV),
                       ysub=torch.from_numpy(Y_SUB[rr]).to(DEV), ys=torch.from_numpy(Y_S[rr]).to(DEV))


# ======================= MODEL =======================
class HierCls(nn.Module):
    """CNN (ResNet / EfficientNetV2 / ConvNeXt) dengan 2 head: season (4) + sub-season (12)."""
    def __init__(self, backbone, grad_ckpt=False, drop=0.3):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=True, num_classes=0)
        if grad_ckpt:
            self.backbone.set_grad_checkpointing(True)   # tukar compute dengan VRAM (~40% lebih hemat)
        d = self.backbone.num_features
        self.drop = nn.Dropout(drop)
        self.head_season, self.head_sub = nn.Linear(d, NUM_SEASONS), nn.Linear(d, NUM_CLASSES)

    def forward(self, x):
        h = self.drop(self.backbone(x))
        return self.head_season(h), self.head_sub(h), None


class DecBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                                  nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="nearest")
        return self.conv(torch.cat([x, skip], 1))


class UNetHier(nn.Module):
    """U-Net (encoder ResNet34) multi-task: segmentasi organ wajah (8 kelas) + klasifikasi hierarkis.
    Fitur klasifikasi = GAP encoder terdalam + region pooling decoder per organ (kulit, rambut, mata, alis, bibir)
    yang dibobot oleh probabilitas segmentasi -> model 'melihat' warna per organ secara eksplisit."""
    REG = [PART_ID[r] for r in ["skin", "hair", "eye", "brow", "lip"]]

    def __init__(self, backbone, dec=(256, 128, 64, 64), drop=0.3, **_):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=True, features_only=True)
        chs = self.backbone.feature_info.channels()
        self.blocks, cin = nn.ModuleList(), chs[-1]
        for skip_c, cout in zip(reversed(chs[:-1]), dec):
            self.blocks.append(DecBlock(cin + skip_c, cout)); cin = cout
        self.seg_head = nn.Conv2d(cin, N_PARTS, 1)
        self.fuse = nn.Sequential(nn.Dropout(drop), nn.Linear(chs[-1] + cin * len(self.REG), 512), nn.GELU(), nn.Dropout(drop))
        self.head_season, self.head_sub = nn.Linear(512, NUM_SEASONS), nn.Linear(512, NUM_CLASSES)

    def forward(self, x):
        feats = self.backbone(x)
        d = feats[-1]
        for blk, skip in zip(self.blocks, reversed(feats[:-1])):
            d = blk(d, skip)
        seg = self.seg_head(d)
        p = seg.float().softmax(1)[:, self.REG]
        w = p / (p.sum((2, 3), keepdim=True) + 1e-4)
        reg = torch.einsum("brhw,bchw->brc", w, d.float()).flatten(1)
        h = self.fuse(torch.cat([feats[-1].float().mean((2, 3)), reg], 1))
        return self.head_season(h), self.head_sub(h), seg


class PatchAttentionNet(nn.Module):
    """Patch organ (kulit pipi, iris, rambut) -> encoder ResNet18 bersama -> Transformer 2 layer antar-organ."""
    def __init__(self, backbone, dim=256, **_):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=True, num_classes=0)
        self.proj = nn.Linear(self.backbone.num_features, dim)
        self.region_emb = nn.Parameter(torch.zeros(1, 3, dim)); self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.missing = nn.Parameter(torch.zeros(1, 1, dim))
        self.attn = nn.TransformerEncoder(nn.TransformerEncoderLayer(dim, 4, dim * 2, dropout=0.1, batch_first=True, norm_first=True), 2)
        self.norm = nn.LayerNorm(dim)
        self.head_season, self.head_sub = nn.Linear(dim, NUM_SEASONS), nn.Linear(dim, NUM_CLASSES)

    def forward(self, x, ok):
        B = x.shape[0]
        t = self.proj(self.backbone(x.flatten(0, 1))).view(B, 3, -1)
        t = torch.where(ok[..., None] > 0, t, self.missing.expand(B, 3, -1).to(t.dtype)) + self.region_emb
        h = self.norm(self.attn(torch.cat([self.cls.expand(B, -1, -1).to(t.dtype), t], 1))[:, 0])
        return self.head_season(h), self.head_sub(h), None


S2S = torch.tensor(SUB2SEASON, device=DEV)


def hierarchical_loss(ls, lsub, ysub, ys):
    l_s = F.cross_entropy(ls.float(), ys)
    l_sub = F.cross_entropy(lsub.float(), ysub, label_smoothing=CFG["label_smoothing"])
    p_s, p_sub = ls.float().softmax(1), lsub.float().softmax(1)
    agg = torch.zeros_like(p_s).index_add_(1, S2S, p_sub)
    return CFG["lambda_season"] * l_s + CFG["lambda_sub"] * l_sub + CFG["lambda_consist"] * (p_s - agg).abs().sum(1).mean()


def compute_loss(out, b):
    ls, lsub, seg = out
    loss = hierarchical_loss(ls, lsub, b["ysub"], b["ys"])
    if seg is not None:
        tgt = F.interpolate(b["m"][:, None].float(), size=seg.shape[-2:], mode="nearest")[:, 0].long()
        ce = F.cross_entropy(seg.float(), tgt, reduction="none").mean((1, 2))
        sv = b["sv"]
        loss = loss + CFG["lambda_seg"] * (ce * sv).sum() / sv.sum().clamp_min(1.0)
    return loss


def combine_heads(ls, lsub):
    # P(sub) = P(sub | head sub) * P(season induk | head season), dinormalisasi
    p = lsub.float().softmax(1) * ls.float().softmax(1)[:, S2S]
    return p / p.sum(1, keepdim=True)


def run_model(model, b):
    return model(b["x"], b["ok"]) if KIND == "patch" else model(b["x"])


def autocast():
    return torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=AMP)


@torch.no_grad()
def predict(model, loader, tta):
    model.eval(); out = []
    for b in loader:
        with autocast():
            o = run_model(model, b); p = combine_heads(o[0], o[1])
            if tta:
                o2 = run_model(model, dict(b, x=torch.flip(b["x"], dims=[-1])))
                p = 0.5 * (p + combine_heads(o2[0], o2[1]))
        out.append(p.float().cpu().numpy())
        del b, o, p
    return np.concatenate(out)


def fake_batch(bs, size):
    if KIND == "patch":
        return dict(x=torch.randn(bs, 3, 3, size, size, device=DEV), ok=torch.ones(bs, 3, device=DEV),
                    ysub=torch.zeros(bs, dtype=torch.long, device=DEV), ys=torch.zeros(bs, dtype=torch.long, device=DEV))
    return dict(x=torch.randn(bs, 3, size, size, device=DEV).contiguous(memory_format=torch.channels_last),
                m=torch.zeros(bs, size, size, dtype=torch.long, device=DEV), sv=torch.ones(bs, device=DEV),
                ysub=torch.zeros(bs, dtype=torch.long, device=DEV), ys=torch.zeros(bs, dtype=torch.long, device=DEV))


def probe_batch(model, target, size):
    """Cari batch fisik terbesar <= target yang memakai < 80% VRAM (sisa untuk optimizer state & augmentasi)."""
    if DEV.type != "cuda":
        return target
    total = torch.cuda.get_device_properties(0).total_memory
    bufs = {k: v.clone() for k, v in model.state_dict().items() if "running" in k or "num_batches" in k}
    bs, chosen = target, 1
    model.train()
    while bs >= 1:
        ok = False
        try:
            torch.cuda.reset_peak_memory_stats()
            b = fake_batch(bs, size)
            with autocast():
                loss = compute_loss(run_model(model, b), b)
            loss.backward(); torch.cuda.synchronize()
            ok = torch.cuda.max_memory_allocated() < 0.80 * total
        except torch.cuda.OutOfMemoryError:
            ok = False
        b = loss = None
        model.zero_grad(set_to_none=True); gc.collect(); torch.cuda.empty_cache()
        if ok:
            chosen = bs; break
        if bs == 1:
            break
        bs = max(1, int(bs * 0.75))
    model.load_state_dict(bufs, strict=False)       # kembalikan statistik BatchNorm (probe memakai data acak)
    return chosen


def slice_batch(b, s):
    return {k: (v[s] if torch.is_tensor(v) else v) for k, v in b.items()}


# ======================= TRAINING =======================
def main():
    t_start = time.time()
    model = {"cls": HierCls, "unet": UNetHier, "patch": PatchAttentionNet}[KIND](CFG["backbone"], grad_ckpt=CFG["grad_ckpt"])
    dcfg = getattr(model.backbone, "pretrained_cfg", None) or {}
    mean, std = dcfg.get("mean", (0.485, 0.456, 0.406)), dcfg.get("std", (0.229, 0.224, 0.225))
    model = model.to(DEV)
    if KIND != "patch" and AMP:
        model = model.to(memory_format=torch.channels_last)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6

    bs = probe_batch(model, CFG["batch"], CFG["img_size"])
    accum = max(1, math.ceil(CFG["batch"] / bs))
    Loader = PatchLoader if KIND == "patch" else MemmapLoader
    dl_tr = Loader(tr_rows, True, bs, CFG["img_size"], mean, std)
    dl_va = Loader(va_rows, False, bs * 2, CFG["img_size"], mean, std)
    dl_te = Loader(test_rows, False, bs * 2, CFG["img_size"], mean, std)
    log(f"{CFG['backbone']} {n_params:.1f}M param | img={CFG['img_size']} | batch fisik={bs} x accum={accum} "
        f"(efektif {bs * accum}) | train={len(tr_rows)} val={len(va_rows)} test={len(test_rows)} | GPU={os.environ.get('CUDA_VISIBLE_DEVICES', '-')}")

    head = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
    body = [p for n, p in model.named_parameters() if n.startswith("backbone.")]
    opt = torch.optim.AdamW([{"params": body, "lr": CFG["lr"]}, {"params": head, "lr": CFG["lr"] * CFG["head_lr_mult"]}],
                            weight_decay=CFG["wd"])
    spe = math.ceil(len(dl_tr) / accum)
    total, warm = CFG["epochs"] * spe, CFG["warmup_epochs"] * spe
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda t: (t + 1) / warm if t < warm else 0.5 * (1 + math.cos(math.pi * min(1.0, (t - warm) / max(1, total - warm)))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)

    n_split, best, best_ep, bad, hist, max_ram, max_vram, ep_times = 1, -1.0, -1, 0, [], 0.0, 0.0, []
    for ep in range(CFG["epochs"]):
        model.train(); t0, run, nb = time.time(), 0.0, 0
        if AMP:
            torch.cuda.reset_peak_memory_stats()
        opt.zero_grad(set_to_none=True)
        for i, b in enumerate(dl_tr):
            B = b["x"].shape[0]
            while True:
                oom, lsum = False, 0.0
                try:
                    step = math.ceil(B / n_split)
                    for s0 in range(0, B, step):
                        sb = slice_batch(b, slice(s0, s0 + step)); frac = sb["x"].shape[0] / B
                        with autocast():
                            loss = compute_loss(run_model(model, sb), sb)
                        scaler.scale(loss * frac / accum).backward()
                        lsum += loss.item() * frac          # .item() -> float Python, graf autograd tidak ikut disimpan
                        del sb, loss
                except torch.cuda.OutOfMemoryError:
                    oom = True
                if not oom:
                    break
                sb = loss = None
                opt.zero_grad(set_to_none=True); gc.collect(); torch.cuda.empty_cache()
                n_split *= 2
                log(f"CUDA OOM -> batch dibelah {n_split} bagian (micro-batch {math.ceil(B / n_split)})")
                if n_split > B:
                    raise RuntimeError("OOM bahkan dengan micro-batch 1")
            run += lsum; nb += 1
            del b
            if (i + 1) % accum == 0 or i + 1 == len(dl_tr):
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                if sched.last_epoch < total:
                    sched.step()
        p_va = predict(model, dl_va, tta=False)
        met = hier_metrics(Y_SUB[va_rows], p_va)
        vram = torch.cuda.max_memory_allocated() / 1e9 if AMP else 0.0
        max_ram, max_vram = max(max_ram, ram_gb()), max(max_vram, vram); dt = time.time() - t0; ep_times.append(dt)
        hist.append(dict(epoch=ep + 1, loss=run / max(nb, 1), **met))
        improved = met["sub_f1"] > best
        log(f"ep {ep + 1:2d}/{CFG['epochs']} loss={run / max(nb, 1):.4f} val season_acc={met['season_acc']:.4f} "
            f"sub_acc={met['sub_acc']:.4f} sub_f1={met['sub_f1']:.4f}{' *' if improved else ''} | "
            f"VRAM {vram:.1f}GB RAM {ram_gb():.2f}GB cgroup {cgroup_gb():.1f}GB | {dt:.0f}s")
        if improved:
            best, best_ep, bad = met["sub_f1"], ep + 1, 0
            torch.save(model.state_dict(), W_PATH)        # langsung ke disk: tidak ada salinan state_dict di RAM
        else:
            bad += 1
            if bad >= CFG["patience"]:
                log(f"early stopping (tidak membaik {bad} epoch)"); break
        del p_va
        release_ram()

    model.load_state_dict(torch.load(W_PATH, map_location=DEV))
    p_va = predict(model, dl_va, tta=True)
    p_te = predict(model, dl_te, tta=True)
    np.save(OUT["oof"], p_va.astype(np.float32)); np.save(OUT["te"], p_te.astype(np.float32))
    met_va = hier_metrics(Y_SUB[va_rows], p_va)
    summary = dict(tag=TAG, model=args.model, family=CFG["family"], backbone=CFG["backbone"], fold=K, img_size=CFG["img_size"],
                   batch_phys=bs, accum=accum, batch_eff=bs * accum, n_split_final=n_split, params_m=round(n_params, 2),
                   best_epoch=best_ep, epochs_run=len(hist), val_tta=met_va,
                   peak_vram_gb=round(max_vram, 2), max_ram_gb=round(max_ram, 2), sec_per_epoch=round(float(np.mean(ep_times)), 1),
                   minutes=round((time.time() - t_start) / 60, 1), history=hist)
    json.dump(summary, open(OUT_JSON, "w"), indent=1, default=float)
    log(f"SELESAI best_ep={best_ep} val(TTA) season_acc={met_va['season_acc']:.4f} sub_f1={met_va['sub_f1']:.4f} "
        f"| {summary['minutes']} menit")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile armo_driver.py
# armo_driver.py — orkestrasi: subprocess per (model, fold) di 2 GPU, LightGBM, ensemble hierarkis, laporan.
# Proses notebook utama TIDAK pernah membuat konteks CUDA / memuat model deep -> RAM-nya tetap kecil sepanjang run.
import os, sys, glob, json, time, shutil, subprocess
import numpy as np, pandas as pd
from armo_common import *

HERE = os.path.dirname(os.path.abspath(__file__))


def child_env(gpu=None):
    env = dict(os.environ, ARMO_BASE=BASE_DIR, ARMO_OUTPUT=OUTPUT_DIR, ARMO_CACHE=CACHE_DIR,
               MALLOC_ARENA_MAX="2",                                # batasi arena glibc -> fragmentasi RAM jauh lebih kecil
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",  # kurangi OOM akibat fragmentasi VRAM
               OMP_NUM_THREADS="2", PYTHONUNBUFFERED="1")
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    return env


def n_gpus():
    try:
        out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout   # tanpa membuat konteks CUDA
        return len([l for l in out.splitlines() if l.startswith("GPU")])
    except FileNotFoundError:
        return 0


def resume_from_inputs():
    """Salin output run sebelumnya yang di-attach lewat 'Add Input -> Your Work' (masker, fitur, prediksi per fold).
    Masker organ dari Versi 2 (folder part_masks) formatnya sama, jadi ikut dipakai ulang."""
    if not IS_KAGGLE:
        return
    for d in sorted(glob.glob("/kaggle/input/**/part_masks", recursive=True)):
        print("resume masker:", d)
        shutil.copytree(d, MASK_CACHE_DIR, dirs_exist_ok=True)
    for d in sorted(glob.glob("/kaggle/input/**/output_v3", recursive=True)):
        print("resume output_v3:", d)
        shutil.copytree(d, OUTPUT_DIR, dirs_exist_ok=True, ignore=shutil.ignore_patterns("_cache", "*.ipynb", "*.html"))


def run_prep():
    t = time.time()
    rc = subprocess.run([sys.executable, os.path.join(HERE, "armo_prep.py")], env=child_env(0 if n_gpus() else None), cwd=HERE).returncode
    if rc != 0:
        raise RuntimeError(f"armo_prep.py gagal (rc={rc})")
    print(f"pre-processing selesai {(time.time() - t) / 60:.1f} menit | RAM notebook {ram_gb():.2f} GB | cgroup {cgroup_gb():.1f} GB")


def job_done(tag, fold):
    return all(os.path.exists(os.path.join(PRED_DIR, f)) for f in [f"{tag}_oof_f{fold}.npy", f"{tag}_te_f{fold}.npy", f"{tag}_f{fold}.json"])


def run_jobs(jobs, poll=5):
    """jobs: list dict(model, fold, [tag, img_size, epochs, batch]). Satu subprocess per GPU, log di-stream ke notebook."""
    gpus = list(range(n_gpus())) or [None]
    pending = [j for j in jobs if not job_done(j.get("tag", j["model"]), j["fold"])]
    print(f"{len(jobs) - len(pending)} job sudah selesai (resume), {len(pending)} job dijalankan di GPU {gpus}")
    running, failed, t0, last_mem = {}, [], time.time(), 0
    while pending or running:
        for g in gpus:
            if g not in running and pending:
                j = pending.pop(0)
                tag = j.get("tag", j["model"])
                cmd = [sys.executable, os.path.join(HERE, "armo_train.py"), "--model", j["model"], "--fold", str(j["fold"])]
                for k in ["tag", "img_size", "epochs", "batch"]:
                    if j.get(k):
                        cmd += [f"--{k}", str(j[k])]
                lf = os.path.join(LOG_DIR, f"{tag}_f{j['fold']}.log")
                fh = open(lf, "w")
                running[g] = (subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=child_env(g), cwd=HERE), j, lf, fh, 0)
        for g, (p, j, lf, fh, pos) in list(running.items()):
            with open(lf, errors="replace") as f:
                f.seek(pos); new = f.read(); pos = f.tell()
            for line in new.splitlines():
                if line.strip() and ("Traceback" in line or "Error" in line or line.startswith("[")):
                    print(f"GPU{g if g is not None else '-'} {line}", flush=True)
            running[g] = (p, j, lf, fh, pos)
            if p.poll() is not None:
                fh.close(); del running[g]
                if p.returncode != 0:
                    failed.append(j)
                    print(f"!! job {j} gagal rc={p.returncode}. 30 baris terakhir {lf}:\n" + "".join(open(lf, errors="replace").readlines()[-30:]))
        if time.time() - last_mem > 300:
            last_mem = time.time()
            print(f"   [monitor {(time.time() - t0) / 60:.0f} mnt] RAM notebook+anak {ram_gb():.1f} GB | cgroup {cgroup_gb():.1f} GB | "
                  f"antrian {len(pending)} | jalan {[r[1]['model'] + '/f' + str(r[1]['fold']) for r in running.values()]}", flush=True)
        time.sleep(poll)
    print(f"semua job selesai dalam {(time.time() - t0) / 60:.1f} menit; gagal: {len(failed)}")
    return failed


def load_meta():
    meta = pd.read_csv(META_CSV, keep_default_na=False)
    tr = np.where(meta.partition.values == "train")[0]
    te = np.where(meta.partition.values == "test")[0]
    return meta, tr, te, meta.fold.values[tr]


# ---------------- Model B: LightGBM kolorimetri (flat 12 kelas & hierarkis season -> expert per season) ----------------
def lgb_model_files(tag, fold):
    """File booster LightGBM per fold (di WEIGHT_DIR). lgb_hier = 1 model season + 1 expert sub-season per season."""
    if tag == "lgb_flat":
        return [f"{tag}_f{fold}.txt"]
    return [f"{tag}_season_f{fold}.txt"] + [f"{tag}_expert{s}_f{fold}.txt" for s in range(NUM_SEASONS)]


def lgb_models_saved(tag, fold):
    return all(os.path.exists(os.path.join(WEIGHT_DIR, f)) for f in lgb_model_files(tag, fold))


def train_lgb_models(meta, tr, te, folds, threads=4):
    import lightgbm as lgb
    feat = pd.read_parquet(FEAT_CACHE)
    assert (feat["key"].values == meta["key"].values).all()
    X = feat.drop(columns="key").values.astype(np.float32)
    y, ys = meta.y_sub.values, meta.y_season.values
    P = dict(objective="multiclass", learning_rate=0.03, num_leaves=31, min_child_samples=15, feature_fraction=0.7,
             bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, seed=SEED, num_threads=threads)

    def fit(Xa, ya, Xv, yv, nc):
        return lgb.train({**P, "num_class": nc}, lgb.Dataset(Xa, ya), 3000, valid_sets=[lgb.Dataset(Xv, yv)],
                         callbacks=[lgb.early_stopping(100, verbose=False)])

    out = {}
    for tag in ["lgb_flat", "lgb_hier"]:
        # dilewati hanya bila prediksi DAN booster tersimpan (output lama dari resume tidak punya booster -> dilatih ulang)
        if all(job_done(tag, k) and lgb_models_saved(tag, k) for k in range(N_FOLDS)):
            continue
        for k in range(N_FOLDS):
            a, v = tr[folds != k], tr[folds == k]
            files = [os.path.join(WEIGHT_DIR, f) for f in lgb_model_files(tag, k)]
            if tag == "lgb_flat":
                m = fit(X[a], y[a], X[v], y[v], NUM_CLASSES)
                m.save_model(files[0], num_iteration=m.best_iteration)
                pv, pt = m.predict(X[v], num_iteration=m.best_iteration), m.predict(X[te], num_iteration=m.best_iteration)
            else:   # P(sub) = P(season) * P(sub | season)  -> 1 model season + 4 expert sub-season
                ms = fit(X[a], ys[a], X[v], ys[v], NUM_SEASONS)
                ms.save_model(files[0], num_iteration=ms.best_iteration)
                pv = np.zeros((len(v), NUM_CLASSES)); pt = np.zeros((len(te), NUM_CLASSES))
                psv, pst = ms.predict(X[v], num_iteration=ms.best_iteration), ms.predict(X[te], num_iteration=ms.best_iteration)
                for s in range(NUM_SEASONS):
                    kids = np.where(SUB2SEASON == s)[0]; loc = {c: i for i, c in enumerate(kids)}
                    aa, vv = a[ys[a] == s], v[ys[v] == s]
                    me = fit(X[aa], [loc[c] for c in y[aa]], X[vv], [loc[c] for c in y[vv]], len(kids))
                    me.save_model(files[1 + s], num_iteration=me.best_iteration)
                    pv[:, kids] = psv[:, [s]] * me.predict(X[v], num_iteration=me.best_iteration)
                    pt[:, kids] = pst[:, [s]] * me.predict(X[te], num_iteration=me.best_iteration)
            np.save(os.path.join(PRED_DIR, f"{tag}_oof_f{k}.npy"), pv.astype(np.float32))
            np.save(os.path.join(PRED_DIR, f"{tag}_te_f{k}.npy"), pt.astype(np.float32))
            json.dump(dict(tag=tag, model=tag, family="COLOR", fold=k, val_tta=hier_metrics(y[v], pv)),
                      open(os.path.join(PRED_DIR, f"{tag}_f{k}.json"), "w"), default=float)
        print(f"{tag}: selesai | booster -> {WEIGHT_DIR}")


# ---------------- pengumpulan prediksi ----------------
def collect(tags, meta, tr, te, folds):
    """-> {tag: dict(oof, te, family, folds=[json...])}; model dengan fold tidak lengkap dilewati (dilaporkan)."""
    res = {}
    for tag in tags:
        if not all(job_done(tag, k) for k in range(N_FOLDS)):
            print(f"  [lewati] {tag}: fold belum lengkap ({sum(job_done(tag, k) for k in range(N_FOLDS))}/{N_FOLDS})"); continue
        oof, tst, js = np.zeros((len(tr), NUM_CLASSES)), np.zeros((len(te), NUM_CLASSES)), []
        for k in range(N_FOLDS):
            oof[folds == k] = np.load(os.path.join(PRED_DIR, f"{tag}_oof_f{k}.npy"))
            tst += np.load(os.path.join(PRED_DIR, f"{tag}_te_f{k}.npy")) / N_FOLDS
            js.append(json.load(open(os.path.join(PRED_DIR, f"{tag}_f{k}.json"))))
        res[tag] = dict(oof=oof, te=tst, family=js[0]["family"], folds=js)
    return res


def temp_scale(p, T):
    lg = np.log(np.clip(p, 1e-9, 1)) / T
    lg -= lg.max(1, keepdims=True); e = np.exp(lg)
    return e / e.sum(1, keepdims=True)


def fit_temperature(p, y):
    from scipy.optimize import minimize_scalar
    from sklearn.metrics import log_loss
    return minimize_scalar(lambda T: log_loss(y, temp_scale(p, T), labels=list(range(NUM_CLASSES))),
                           bounds=(0.3, 10.0), method="bounded").x


# ---------------- ensemble hierarkis: fold -> model -> family -> final ----------------
def build_ensemble(res, y_tr, y_te, skin_b_tr, skin_b_te, trials=300):
    import optuna
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_predict, StratifiedKFold
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    for r in res.values():                                     # level 1: kalibrasi per model
        r["T"] = fit_temperature(r["oof"], y_tr)
        r["oof_c"], r["te_c"] = temp_scale(r["oof"], r["T"]), temp_scale(r["te"], r["T"])
    fams = {}
    for tag, r in res.items():                                 # level 2: rata-rata dalam family
        fams.setdefault(r["family"], []).append(tag)
    FAM = {f: dict(members=t, oof=np.mean([res[x]["oof_c"] for x in t], 0), te=np.mean([res[x]["te_c"] for x in t], 0))
           for f, t in fams.items()}
    names = list(FAM)

    cands = {}                                                 # level 3: kandidat penggabung antar family
    cands["Soft voting rata (family)"] = (np.mean([FAM[f]["oof"] for f in names], 0), np.mean([FAM[f]["te"] for f in names], 0), None)

    def obj(trial):
        w = np.array([trial.suggest_float(f, 1e-3, 1.0) for f in names]); w /= w.sum()
        return hier_metrics(y_tr, sum(wi * FAM[f]["oof"] for wi, f in zip(w, names)))["sub_f1"]
    st = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    st.optimize(obj, n_trials=trials)
    w = np.array([st.best_params[f] for f in names]); w /= w.sum()
    W = {f: round(float(x), 4) for f, x in zip(names, w)}
    cands["Soft voting Optuna (family)"] = (sum(wi * FAM[f]["oof"] for wi, f in zip(w, names)),
                                            sum(wi * FAM[f]["te"] for wi, f in zip(w, names)), W)

    Z_tr = np.concatenate([np.log(np.clip(res[t]["oof_c"], 1e-6, 1)) for t in res], 1)   # stacking meta-learner
    Z_te = np.concatenate([np.log(np.clip(res[t]["te_c"], 1e-6, 1)) for t in res], 1)
    lr = LogisticRegression(C=0.1, max_iter=3000)
    p_st = cross_val_predict(lr, Z_tr, y_tr, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")
    cands["Stacking LogReg (semua model)"] = (p_st, lr.fit(Z_tr, y_tr).predict_proba(Z_te), None)

    best_name = max(cands, key=lambda c: hier_metrics(y_tr, cands[c][0])["sub_f1"])
    P_oof, P_te, _ = cands[best_name]

    # Chromatic arbiter (warm vs cool) — hanya dipakai bila menaikkan F1 OOF
    IS_WARM = np.array([SUB2SEASON[k] in WARM_SEASONS for k in range(NUM_CLASSES)])

    def arbiter(p, b, margin, tw, tc):
        s_hat, sub = hierarchical_decode(p)
        o = np.argsort(-p, 1); t1, t2 = o[:, 0], o[:, 1]
        gap = p[np.arange(len(p)), t1] - p[np.arange(len(p)), t2]
        con = (IS_WARM[t1] != IS_WARM[t2]) & (gap < margin)
        wc, cc = np.where(IS_WARM[t1], t1, t2), np.where(IS_WARM[t1], t2, t1)
        sub = sub.copy(); sub[con & (b > tw)] = wc[con & (b > tw)]; sub[con & (b <= tc)] = cc[con & (b <= tc)]
        return SUB2SEASON[sub], sub
    q = np.percentile(skin_b_tr, [10, 50, 90]); base = hier_metrics(y_tr, P_oof)["sub_f1"]

    def obj_a(trial):
        m = trial.suggest_float("margin", 0.0, 0.15); tc = trial.suggest_float("t_cool", q[0], q[1]); tw = trial.suggest_float("t_warm", tc, q[2])
        return hier_metrics(y_tr, None, *arbiter(P_oof, skin_b_tr, m, tw, tc))["sub_f1"]
    sa = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    sa.optimize(obj_a, n_trials=trials)
    use_arb = sa.best_value > base + 1e-3
    A = sa.best_params
    if use_arb:
        dec_oof = arbiter(P_oof, skin_b_tr, A["margin"], A["t_warm"], A["t_cool"])
        dec_te = arbiter(P_te, skin_b_te, A["margin"], A["t_warm"], A["t_cool"])
    else:
        dec_oof, dec_te = hierarchical_decode(P_oof), hierarchical_decode(P_te)
    final_name = best_name + (" + chromatic arbiter" if use_arb else "")
    return dict(FAM=FAM, cands=cands, best=best_name, final_name=final_name, weights=W, use_arbiter=use_arb, arbiter=A,
                P_oof=P_oof, P_te=P_te, dec_oof=dec_oof, dec_te=dec_te,
                temps={t: round(float(r["T"]), 3) for t, r in res.items()})


def report_tree(res, ens, y_tr, y_te):
    """Tabel hierarki: FINAL -> kandidat -> family -> model -> fold (OOF = validasi silang train, TEST = official test)."""
    rows = []

    def add(level, name, p_oof, p_te, dec_oof=None, dec_te=None, extra=""):
        mo = hier_metrics(y_tr, p_oof, *(dec_oof or (None, None)))
        mt = hier_metrics(y_te, p_te, *(dec_te or (None, None)))
        rows.append(dict(level=level, name=name, oof_sub_f1=mo["sub_f1"], oof_season_acc=mo["season_acc"],
                         test_season_acc=mt["season_acc"], test_season_f1=mt["season_f1"], test_sub_acc=mt["sub_acc"],
                         test_sub_f1=mt["sub_f1"], test_cond_sub_acc=mt["cond_sub_acc"], info=extra))
    add(0, "FINAL: " + ens["final_name"], ens["P_oof"], ens["P_te"], ens["dec_oof"], ens["dec_te"])
    for c, (po, pt, w) in ens["cands"].items():
        add(1, c, po, pt, extra=f"bobot={w}" if w else "")
    for f, F in ens["FAM"].items():
        add(1, f"[{f}] family mean", F["oof"], F["te"], extra=f"anggota={F['members']}")
        for tag in F["members"]:
            r = res[tag]
            add(2, tag, r["oof_c"], r["te_c"], extra=f"T={r['T']:.2f}")
            for j in r["folds"]:
                v = j["val_tta"]
                rows.append(dict(level=3, name=f"{tag} / fold {j['fold']}", oof_sub_f1=v["sub_f1"], oof_season_acc=v["season_acc"],
                                 info=(f"img={j.get('img_size')} batch={j.get('batch_phys')}x{j.get('accum')} best_ep={j.get('best_epoch')} "
                                       f"VRAM={j.get('peak_vram_gb')}GB RAM={j.get('max_ram_gb')}GB {j.get('sec_per_epoch')}s/ep"
                                       if "img_size" in j else "")))
    df = pd.DataFrame(rows)
    pre = {0: "", 1: "├─ ", 2: "│   ├─ ", 3: "│   │   └─ "}
    print(f"{'HIERARKI MODEL':58s} {'OOF subF1':>9s} {'OOF seaAcc':>10s} | {'TEST seaAcc':>11s} {'subAcc':>7s} {'subF1':>7s} {'condAcc':>7s}")
    print("-" * 118)
    for r in df.itertuples():
        fmt = lambda v: f"{v:.4f}" if isinstance(v, float) and np.isfinite(v) else "   -  "
        print(f"{(pre[r.level] + r.name)[:58]:58s} {fmt(r.oof_sub_f1):>9s} {fmt(r.oof_season_acc):>10s} | "
              f"{fmt(r.test_season_acc):>11s} {fmt(r.test_sub_acc):>7s} {fmt(r.test_sub_f1):>7s} {fmt(r.test_cond_sub_acc):>7s}  {r.info}")
    return df


def report_recommendation(res, y_te):
    rows = []
    for tag, r in res.items():
        js = r["folds"]
        if "img_size" not in js[0]:
            continue
        mt = hier_metrics(y_te, r["te_c"])
        rows.append(dict(model=tag, family=r["family"], backbone=js[0]["backbone"], img_size=js[0]["img_size"],
                         batch_fisik=int(np.min([j["batch_phys"] for j in js])), accum=js[0]["accum"],
                         batch_efektif=js[0]["batch_eff"], peak_vram_gb=max(j["peak_vram_gb"] for j in js),
                         max_ram_gb=max(j["max_ram_gb"] for j in js), sec_per_epoch=np.mean([j["sec_per_epoch"] for j in js]),
                         menit_total=sum(j["minutes"] for j in js), test_season_acc=mt["season_acc"], test_sub_acc=mt["sub_acc"],
                         test_sub_f1=mt["sub_f1"]))
    df = pd.DataFrame(rows).sort_values("test_sub_f1", ascending=False)
    if len(df):
        b = df.iloc[0]
        print("\n=== REKOMENDASI (diukur dari run ini) ===")
        print(f"Model deep tunggal terbaik : {b.model} ({b.backbone})")
        print(f"  image size               : {b.img_size} px")
        print(f"  batch                    : fisik {b.batch_fisik} x accum {b.accum} = efektif {b.batch_efektif}")
        print(f"  VRAM puncak / RAM proses : {b.peak_vram_gb:.1f} GB / {b.max_ram_gb:.1f} GB")
        print(f"  TEST season accuracy     : {b.test_season_acc:.4f}")
        print(f"  TEST sub-season accuracy : {b.test_sub_acc:.4f}")
        print(f"  TEST sub-season macro F1 : {b.test_sub_f1:.4f}")
    return df


# ---------------- rincian per season & sub-season ----------------
SEASON_THEORY = {
    "autunno": "warm undertone, value sedang-gelap, chroma rendah-sedang (earthy)",
    "estate": "cool undertone, value terang-sedang, chroma rendah (lembut, berdebu)",
    "inverno": "cool undertone, kontras tinggi, chroma tinggi (tegas, jernih)",
    "primavera": "warm undertone, value terang, chroma tinggi (cerah, segar)",
}
SUB_THEORY = {
    "autunno-deep": "gelap & hangat, kontras sedang-tinggi", "autunno-soft": "hangat-netral, chroma rendah, kontras rendah",
    "autunno-warm": "paling hangat (b* kulit tinggi), chroma sedang",
    "estate-cool": "paling dingin, chroma rendah-sedang", "estate-light": "terang & dingin, kontras rendah",
    "estate-soft": "dingin-netral, chroma rendah, kontras rendah",
    "inverno-bright": "chroma sangat tinggi, kontras tinggi", "inverno-cool": "paling dingin, kontras tinggi",
    "inverno-deep": "gelap & dingin, kontras tinggi",
    "primavera-bright": "hangat & chroma tinggi, kontras sedang", "primavera-light": "paling terang, hangat, kontras rendah",
    "primavera-warm": "paling hangat (b* kulit tinggi), terang-sedang",
}


def report_per_class(res, ens, meta, tr, te):
    """Per kelas: (a) profil kolorimetri data train vs teori, (b) recall/precision/F1/support model FINAL,
    (c) akurasi per kelas setiap model (siapa yang paling jago di kelas mana), (d) kebingungan terbesar."""
    from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
    y_tr, y_te = meta.y_sub.values[tr], meta.y_sub.values[te]
    s_te = SUB2SEASON[y_te]
    s_fin, sub_fin = ens["dec_te"]
    out = {}

    # (a) profil kolorimetri (median train) — menjelaskan KENAPA kelas berbeda
    feat = pd.read_parquet(FEAT_CACHE).drop(columns="key").iloc[tr]
    cols = [c for c in ["skin_L", "skin_b", "skin_C", "ITA", "hair_L", "eye_L", "dL_skin_hair", "contrast_max"] if c in feat]
    prof_sub = feat[cols].groupby([SUB_NAMES[i] for i in y_tr]).median().reindex(SUB_NAMES)
    prof_sub.insert(0, "teori", [SUB_THEORY[n] for n in SUB_NAMES])
    prof_sea = feat[cols].groupby([SEASONS[i] for i in SUB2SEASON[y_tr]]).median().reindex(SEASONS)
    prof_sea.insert(0, "teori", [SEASON_THEORY[n] for n in SEASONS])
    out["profil_season"], out["profil_sub"] = prof_sea, prof_sub

    # (b) metrik per kelas model FINAL
    def per_class(y, yhat, names):
        p, r, f, n = precision_recall_fscore_support(y, yhat, labels=list(range(len(names))), zero_division=0)
        return pd.DataFrame(dict(support=n, accuracy_recall=r, precision=p, f1=f), index=names)
    out["final_season"] = per_class(s_te, s_fin, SEASONS)
    fs = per_class(y_te, sub_fin, SUB_NAMES)
    ok_s = s_fin == s_te
    fs.insert(1, "season_benar", [float(ok_s[y_te == k].mean()) if (y_te == k).any() else np.nan for k in range(NUM_CLASSES)])
    out["final_sub"] = fs

    # (c) akurasi (recall) per kelas untuk setiap model + FINAL
    acc_sea, acc_sub = {}, {}
    for tag, r in res.items():
        sh, bh = hierarchical_decode(r["te_c"])
        acc_sea[tag] = [float((sh[s_te == s] == s).mean()) for s in range(NUM_SEASONS)]
        acc_sub[tag] = [float((bh[y_te == k] == k).mean()) if (y_te == k).any() else np.nan for k in range(NUM_CLASSES)]
    acc_sea["FINAL"] = out["final_season"]["accuracy_recall"].tolist()
    acc_sub["FINAL"] = out["final_sub"]["accuracy_recall"].tolist()
    out["acc_season_per_model"] = pd.DataFrame(acc_sea, index=SEASONS)
    out["acc_sub_per_model"] = pd.DataFrame(acc_sub, index=SUB_NAMES)
    m = out["acc_sub_per_model"].drop(columns="FINAL")
    out["acc_sub_per_model"]["model_terbaik"] = m.idxmax(1)

    # (d) kebingungan terbesar
    cm = confusion_matrix(y_te, sub_fin, labels=range(NUM_CLASSES)).astype(float)
    np.fill_diagonal(cm, 0)
    pairs = [(SUB_NAMES[i], SUB_NAMES[j], int(cm[i, j]), cm[i, j] / max((y_te == i).sum(), 1),
              "season sama" if SUB2SEASON[i] == SUB2SEASON[j] else
              ("warm<->cool" if (SUB2SEASON[i] in WARM_SEASONS) != (SUB2SEASON[j] in WARM_SEASONS) else "season beda, temperatur sama"))
             for i, j in zip(*np.unravel_index(np.argsort(-cm, axis=None)[:10], cm.shape)) if cm[i, j] > 0]
    out["kebingungan"] = pd.DataFrame(pairs, columns=["label_asli", "diprediksi_sebagai", "jumlah", "porsi_dari_kelas", "jenis_kesalahan"])

    # ringkasan teks per season -> sub-season
    print("=" * 100); print("RINCIAN PER SEASON & SUB-SEASON — model FINAL pada official test"); print("=" * 100)
    for s, sn in enumerate(SEASONS):
        r = out["final_season"].loc[sn]
        print(f"\n■ {sn.upper():10s} acc={r.accuracy_recall:.3f} prec={r.precision:.3f} F1={r.f1:.3f} (n={int(r.support)}) — {SEASON_THEORY[sn]}")
        for k in np.where(SUB2SEASON == s)[0]:
            n = SUB_NAMES[k]; q = out["final_sub"].loc[n]; pr = prof_sub.loc[n]
            wrong = out["kebingungan"][out["kebingungan"].label_asli == n]
            tip = f" | paling sering tertukar dgn {wrong.iloc[0].diprediksi_sebagai} ({wrong.iloc[0].jumlah}x)" if len(wrong) else ""
            print(f"   └─ {n:17s} acc={q.accuracy_recall:.3f} F1={q.f1:.3f} season-benar={q.season_benar:.3f} n={int(q.support):3d} "
                  f"| model terbaik: {out['acc_sub_per_model'].loc[n, 'model_terbaik']:10s} "
                  f"| kulit L*={pr.get('skin_L', np.nan):.1f} b*={pr.get('skin_b', np.nan):.1f} ITA={pr.get('ITA', np.nan):.0f}{tip}")
    return out

In [ ]:
import os, sys, subprocess
for _mod, _pkg in [("timm", "timm"), ("lightgbm", "lightgbm"), ("optuna", "optuna"), ("pyarrow", "pyarrow"), ("psutil", "psutil")]:
    try:
        __import__(_mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=False)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from armo_common import *
from armo_driver import *
print(f"IS_KAGGLE={IS_KAGGLE} | SMOKE={SMOKE}\nBASE_DIR={BASE_DIR}\nOUTPUT_DIR={OUTPUT_DIR}\nCACHE_DIR={CACHE_DIR}")
print("GPU:", subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout
      if n_gpus() else "tidak ada (CPU)")
print(f"RAM notebook {ram_gb():.2f} GB | cgroup {cgroup_gb():.1f} GB")
resume_from_inputs()

## 1. Pre-processing (subprocess)
Langkah-langkah: anotasi → fold stratified → masker organ (facer asli → pyfacer FaRL → heuristik) → fitur RGB/HSV/CIELab/ITA/kontras + patch organ → cache uint8 (crop ke bbox wajah, letterbox persegi 352 px).
Saat subprocess selesai, model pyfacer (~650 MB) ikut dibebaskan sepenuhnya.

In [ ]:
run_prep()
meta, tr, te, folds = load_meta()
y_tr, y_te = meta.y_sub.values[tr], meta.y_sub.values[te]
display(pd.crosstab(meta["y_sub"].map(dict(enumerate(SUB_NAMES))), meta["partition"], margins=True))
display(meta["mask_src"].value_counts().rename("sumber masker"))

In [ ]:
# Cek visual hasil pre-processing (dibaca dari memmap; tidak memuat seluruh array ke RAM)
PALETTE = np.array([(0, 0, 0), (255, 200, 150), (120, 60, 20), (0, 0, 255), (0, 255, 0), (255, 0, 0), (255, 255, 0), (128, 0, 128)], np.uint8)
cp = img_cache_paths(len(meta)); A, O, L = (np.load(cp[k], mmap_mode="r") for k in ["rgbm", "orig", "lm"])
idx = np.random.RandomState(1).choice(len(meta), 6, replace=False)
fig, ax = plt.subplots(3, 6, figsize=(18, 9))
for j, i in enumerate(idx):
    ax[0, j].imshow(A[i]); ax[0, j].set_title(f"{SUB_NAMES[meta.y_sub[i]]}\nRGB-M (CNN)")
    ax[1, j].imshow(O[i]); ax[1, j].set_title("ORIGINAL (U-Net)")
    ax[2, j].imshow(PALETTE[L[i]]); ax[2, j].set_title(f"labelmap: {meta.mask_src[i]}")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show(); del A, O, L

## 2. Model B: LightGBM pada fitur kolorimetri (flat & hierarkis), dijalankan di CPU

In [ ]:
train_lgb_models(meta, tr, te, folds)

## 3. Model deep: augmentasi GPU + training per fold (subprocess, 2 GPU paralel)
Ubah `MODELS` untuk memilih model. Perkiraan waktu di 2×T4: ±3–4 jam untuk 5 model × 5 fold.
Log lengkap per job ada di `OUTPUT_DIR/logs/`. Tiap 5 menit monitor mencetak RAM/cgroup supaya kebocoran langsung terlihat.

In [ ]:
MODELS = ["resnet50", "effnetv2_s", "convnext_t", "unet_r34", "patch_attn"]
jobs = [dict(model=m, fold=k) for k in range(N_FOLDS) for m in MODELS]   # urut per fold -> hasil awal semua model cepat tersedia
failed = run_jobs(jobs)

### 3b. (Opsional) Sweep image size
Membandingkan image size pada fold 0 (6 epoch) dengan backbone yang sama, untuk menjawab berapa image size yang terbaik.
Default dimatikan (±30 menit). Ubah `RUN_SWEEP = True` untuk menjalankannya.

In [ ]:
RUN_SWEEP = False
SWEEP_MODEL, SWEEP_SIZES, SWEEP_EPOCHS = "resnet50", [224, 288, 320, 352], 6
if RUN_SWEEP:
    run_jobs([dict(model=SWEEP_MODEL, fold=0, img_size=s, epochs=SWEEP_EPOCHS, tag=f"sweep_{SWEEP_MODEL}_{s}") for s in SWEEP_SIZES])
    sw = []
    for s in SWEEP_SIZES:
        j = json.load(open(os.path.join(PRED_DIR, f"sweep_{SWEEP_MODEL}_{s}_f0.json")))
        sw.append(dict(img_size=s, batch=f"{j['batch_phys']}x{j['accum']}", vram_gb=j["peak_vram_gb"], sec_per_epoch=j["sec_per_epoch"],
                       val_season_acc=j["val_tta"]["season_acc"], val_sub_f1=j["val_tta"]["sub_f1"]))
    sw = pd.DataFrame(sw); display(sw)
    print("Image size terbaik (val sub F1, fold 0):", int(sw.loc[sw.val_sub_f1.idxmax(), "img_size"]))

## 4. Ensemble hierarkis: fold → model → family → final
Langkah: (1) temperature scaling per model pada OOF, (2) rata-rata dalam family, (3) tiga kandidat penggabung antar family.
Kandidat dengan F1 sub-season OOF terbaik dipilih otomatis. *Chromatic arbiter* (warm vs cool berdasarkan b* kulit) hanya dipakai bila menaikkan F1 OOF.

In [ ]:
TAGS = [t for t in MODELS] + ["lgb_flat", "lgb_hier"]
res = collect(TAGS, meta, tr, te, folds)
feat = pd.read_parquet(FEAT_CACHE)
SKIN_B = np.nan_to_num(feat["skin_b"].values, nan=np.nanmedian(feat["skin_b"].values))
ens = build_ensemble(res, y_tr, y_te, SKIN_B[tr], SKIN_B[te], trials=30 if SMOKE else 300)
print("Temperatur:", ens["temps"]); print("Bobot family (Optuna):", ens["weights"])
print("Arbiter dipakai:", ens["use_arbiter"], {k: round(v, 3) for k, v in ens["arbiter"].items()}, "\n")
tree_df = report_tree(res, ens, y_tr, y_te)

## 5. Hasil akhir: metode, batch size, image size terbaik + evaluasi official test

In [ ]:
rec_df = report_recommendation(res, y_te)
display(rec_df.style.format(precision=3).background_gradient(subset=["test_sub_f1"], cmap="Greens"))

s_final, sub_final = ens["dec_te"]
FINAL = hier_metrics(y_te, ens["P_te"], s_final, sub_final)
print("\n" + "=" * 70)
print(f"METODE TERBAIK : {ens['final_name']}")
print(f"Anggota        : {', '.join(res)}")
print(f"OFFICIAL TEST  : season acc {FINAL['season_acc']:.4f} | season F1 {FINAL['season_f1']:.4f} | "
      f"sub acc {FINAL['sub_acc']:.4f} | sub F1 {FINAL['sub_f1']:.4f} | cond sub acc {FINAL['cond_sub_acc']:.4f}")
print("=" * 70)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
fig, ax = plt.subplots(1, 2, figsize=(22, 9), gridspec_kw={"width_ratios": [1, 2.2]})
for a, cm, names, t in [(ax[0], confusion_matrix(SUB2SEASON[y_te], s_final, labels=range(NUM_SEASONS)), SEASONS, "Season 4x4"),
                        (ax[1], confusion_matrix(y_te, sub_final, labels=range(NUM_CLASSES)), SUB_NAMES, "Sub-season 12x12")]:
    cmn = cm / cm.sum(1, keepdims=True).clip(1)
    a.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(names)):
        for j in range(len(names)):
            a.text(j, i, f"{cm[i, j]}\n{cmn[i, j]:.0%}", ha="center", va="center", fontsize=8, color="white" if cmn[i, j] > 0.5 else "black")
    a.set_xticks(range(len(names))); a.set_xticklabels(names, rotation=60, ha="right"); a.set_yticks(range(len(names))); a.set_yticklabels(names)
    a.set_xlabel("Prediksi"); a.set_ylabel("Label"); a.set_title(t)
for b in [2.5, 5.5, 8.5]:
    ax[1].axhline(b, color="red", lw=1.5); ax[1].axvline(b, color="red", lw=1.5)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrices.png"), dpi=120); plt.show()
print(classification_report(y_te, sub_final, labels=range(NUM_CLASSES), target_names=SUB_NAMES, digits=4, zero_division=0))
print(classification_report(SUB2SEASON[y_te], s_final, labels=range(NUM_SEASONS), target_names=SEASONS, digits=4, zero_division=0))

pd.DataFrame({"key": meta.key.values[te], "y_true": [SUB_NAMES[i] for i in y_te], "y_pred": [SUB_NAMES[i] for i in sub_final],
              "season_pred": [SEASONS[i] for i in s_final], "confidence": ens["P_te"][np.arange(len(te)), sub_final]}
             ).to_csv(os.path.join(OUTPUT_DIR, "test_predictions.csv"), index=False)
tree_df.to_csv(os.path.join(OUTPUT_DIR, "results_hierarchy.csv"), index=False)
rec_df.to_csv(os.path.join(OUTPUT_DIR, "recommendation.csv"), index=False)
json.dump({"final_method": ens["final_name"], "weights": ens["weights"], "temps": ens["temps"], "use_arbiter": bool(ens["use_arbiter"]),
           "arbiter": ens["arbiter"], "final": FINAL, "model_zoo": MODEL_ZOO},
          open(os.path.join(OUTPUT_DIR, "ensemble_config.json"), "w"), indent=2, default=float)
print("tersimpan di", OUTPUT_DIR)

## 5b. Rincian per season & sub-season
1. **Ringkasan teks** per season, lalu per sub-season: akurasi, F1, apakah season-nya sudah benar, model terbaik untuk kelas itu, profil warna kulit, dan kelas yang paling sering tertukar.
2. **Profil kolorimetri** (median data train) dibandingkan dengan teori Armocromia, untuk menjelaskan *kenapa* kelas berbeda.
3. **Akurasi per kelas untuk setiap model** dan heatmap-nya (model mana unggul di kelas mana).
4. **10 kebingungan terbesar**, dengan label jenisnya (season sama / warm↔cool / season beda).

In [ ]:
pc = report_per_class(res, ens, meta, tr, te)

print("\n--- Profil kolorimetri per SEASON (median train) ---");     display(pc["profil_season"].round(2))
print("--- Profil kolorimetri per SUB-SEASON (median train) ---");   display(pc["profil_sub"].round(2))
print("--- Metrik per SEASON (FINAL, test) ---");                    display(pc["final_season"].style.format(precision=4).background_gradient(subset=["accuracy_recall", "f1"], cmap="Greens"))
print("--- Metrik per SUB-SEASON (FINAL, test) ---");                display(pc["final_sub"].style.format(precision=4).background_gradient(subset=["accuracy_recall", "f1"], cmap="Greens"))
print("--- Akurasi per SEASON untuk setiap model (test) ---");       display(pc["acc_season_per_model"].style.format(precision=3).background_gradient(axis=1, cmap="Greens"))
print("--- Akurasi per SUB-SEASON untuk setiap model (test) ---");   display(pc["acc_sub_per_model"].style.format(precision=3, subset=pc["acc_sub_per_model"].columns[:-1]).background_gradient(axis=1, cmap="Greens", subset=pc["acc_sub_per_model"].columns[:-1]))
print("--- 10 kebingungan terbesar (FINAL, test) ---");             display(pc["kebingungan"].round(3))

H = pc["acc_sub_per_model"].drop(columns="model_terbaik")
fig, ax = plt.subplots(figsize=(1.1 * H.shape[1] + 4, 7))
im = ax.imshow(H.values, cmap="Greens", vmin=0, vmax=1, aspect="auto")
for i in range(H.shape[0]):
    for j in range(H.shape[1]):
        ax.text(j, i, f"{H.values[i, j]:.2f}", ha="center", va="center", fontsize=8, color="white" if H.values[i, j] > 0.6 else "black")
ax.set_xticks(range(H.shape[1])); ax.set_xticklabels(H.columns, rotation=45, ha="right"); ax.set_yticks(range(H.shape[0])); ax.set_yticklabels(H.index)
for b in [2.5, 5.5, 8.5]:
    ax.axhline(b, color="red", lw=1.2)
ax.set_title("Akurasi per sub-season untuk setiap model (official test)"); plt.colorbar(im, ax=ax, fraction=0.03)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "per_class_accuracy.png"), dpi=120); plt.show()

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "rincian_per_kelas.xlsx")) as xw:
    for k, v in pc.items():
        v.to_excel(xw, sheet_name=k[:31])
print("tersimpan:", os.path.join(OUTPUT_DIR, "rincian_per_kelas.xlsx"))

## 5c. Simpan hasil ke `.pkl` dan `.joblib`
Sel ini hanya MENYIMPAN objek yang sudah ada di memori (`res`, `ens`, `FINAL`, dst). Tidak ada sel lain atau file `armo_*.py` yang diubah, jadi hasil model tetap sama.

In [ ]:
# ============================================================
# 5c. SIMPAN HASIL KE .pkl (hanya menyimpan, tidak mengubah model / hasil)
# ============================================================
import joblib
from sklearn.linear_model import LogisticRegression

# Meta-learner stacking: di build_ensemble() modelnya tidak dikembalikan, jadi di sini di-fit ulang dengan
# input & parameter yang IDENTIK (LogisticRegression(C=0.1, max_iter=3000) pada log-prob terkalibrasi) -> model sama persis.
_Z_tr = np.concatenate([np.log(np.clip(res[t]["oof_c"], 1e-6, 1)) for t in res], 1)
stack_lr = LogisticRegression(C=0.1, max_iter=3000).fit(_Z_tr, y_tr)

PKL_PATH = os.path.join(OUTPUT_DIR, "armo_v3_hasil.pkl")
hasil = {
    "class_names": SUB_NAMES, "seasons": SEASONS, "subclasses": SUBCLASSES, "sub2season": SUB2SEASON,
    "models": list(res), "model_zoo": {t: MODEL_ZOO[t] for t in res if t in MODEL_ZOO},
    "temps": {t: float(r["T"]) for t, r in res.items()},
    "families": {f: v["members"] for f, v in ens["FAM"].items()},
    "best_combiner": ens["best"], "final_name": ens["final_name"],
    "optuna_family_weights": ens["weights"], "stack_lr": stack_lr, "stack_tags": list(res),
    "use_arbiter": bool(ens["use_arbiter"]), "arbiter_params": ens["arbiter"],
    "pred_oof": {t: r["oof"] for t, r in res.items()}, "pred_test": {t: r["te"] for t, r in res.items()},
    "P_oof": ens["P_oof"], "P_te": ens["P_te"], "dec_te": ens["dec_te"],
    "test_keys": meta["key"].values[te], "y_te": y_te,
    "official_test_metrics": {k: float(v) for k, v in FINAL.items()},
    "deep_ckpts": {t: [os.path.join(WEIGHT_DIR, f"{t}_f{k}.pt") for k in range(N_FOLDS)] for t in res if t in MODEL_ZOO},
    # LightGBM: path RELATIF terhadap OUTPUT_DIR (file .txt, muat dengan lgb.Booster(model_file=...)).
    # lgb_flat: [fold] -> 1 file; lgb_hier: [fold] -> [season, expert0..expert{NUM_SEASONS-1}]
    "lgb_models": {t: [[os.path.join("weights", f) for f in lgb_model_files(t, k)] for k in range(N_FOLDS)]
                   for t in res if t.startswith("lgb_")},
    "lgb_feature_cols": [c for c in pd.read_parquet(FEAT_CACHE).columns if c != "key"],   # urutan kolom fitur input
    "lgb_hier_kids": {s: np.where(SUB2SEASON == s)[0].tolist() for s in range(NUM_SEASONS)},  # expert s -> indeks sub-kelas
}
_missing = [f for t in hasil["lgb_models"] for fs in hasil["lgb_models"][t] for f in fs
            if not os.path.exists(os.path.join(OUTPUT_DIR, f))]
assert not _missing, f"booster LightGBM belum tersimpan: {_missing[:3]} ... -> jalankan ulang train_lgb_models"
JOBLIB_PATH = os.path.join(OUTPUT_DIR, "armo_v3_hasil.joblib")
joblib.dump(hasil, PKL_PATH)
joblib.dump(hasil, JOBLIB_PATH)
for _f in [PKL_PATH, JOBLIB_PATH]:
    print(f"Tersimpan: {_f} ({os.path.getsize(_f) / 1e6:.2f} MB)")


## 6. Bersihkan cache & catat ke `modellab_registry` (notebook + catatan hasil -> repo GitHub)

In [ ]:
# ============================================================
# 6. CATAT HASIL KE modellab_registry (notebook + catatan hasil -> repo GitHub)
# ============================================================
# Sel ini HANYA membaca hasil yang sudah dihitung di sel sebelumnya (FINAL, ens, y_te, sub_final).
# Tidak ada model / preprocessing / training yang dijalankan ulang.
import shutil, json
if CACHE_DIR.startswith(OUTPUT_DIR):            # cache memmap jangan ikut tersimpan sebagai output Kaggle
    shutil.rmtree(CACHE_DIR, ignore_errors=True)

# Utility script Kaggle "modellab_registry" (Add Input -> Utility Script) dipasang di /kaggle/usr/lib/...
for _p in [os.getcwd(), "/kaggle/working",
           *sorted({os.path.dirname(p) for p in glob.glob("/kaggle/usr/lib/**/modellab_registry.py", recursive=True)}),
           *sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/modellab_registry.py", recursive=True)})]:
    if os.path.exists(os.path.join(_p, "modellab_registry.py")) and _p not in sys.path:
        sys.path.insert(0, _p)

LOG_TO_REGISTRY = not SMOKE

# 0. Kumpulkan metrik official test (sub-season 12 kelas = metrik utama, sama seperti eksperimen topic-a lain)
from sklearn.metrics import precision_recall_fscore_support
_prec, _rec, _, _ = precision_recall_fscore_support(y_te, sub_final, labels=list(range(NUM_CLASSES)), average="macro", zero_division=0)
_top3 = float(np.mean([y_te[i] in np.argsort(-ens["P_te"][i])[:3] for i in range(len(y_te))]))
final_metrics = {"accuracy": float(FINAL["sub_acc"]), "macro_f1": float(FINAL["sub_f1"]), "macro_precision": float(_prec),
                 "macro_recall": float(_rec), "top3": _top3,
                 "season_accuracy": float(FINAL["season_acc"]), "season_f1": float(FINAL["season_f1"]),
                 "cond_sub_accuracy": float(FINAL["cond_sub_acc"])}
print({k: round(float(v), 4) for k, v in final_metrics.items()})

# 1. Ekstrak hasil evaluasi yang sudah dihitung oleh notebook di sel sebelumnya
accuracy = final_metrics["accuracy"]
f1_score = final_metrics["macro_f1"]
precision = final_metrics["macro_precision"]
recall = final_metrics["macro_recall"]

REGISTRY_INFO = dict(
    topic="topic-a",
    model=f"Hierarchical Ensemble v3 ({', '.join(res)}) -> {ens['final_name']}",
    objective="Deteksi season (4) & sub-season (12) Armocromia, subprocess per fold (bebas OOM), evaluasi official test",
    dataset="deep_armocromia",
    owner="Fatih Itsbatul Haq",
)

# 2. Panggil fungsi auto-log, sertakan globals() dan panggil variabel konfigurasi master
if not LOG_TO_REGISTRY:
    print("SMOKE=True -> tidak dicatat ke registry")
else:
    try:
        import modellab_registry
    except ImportError:
        modellab_registry = None
    if modellab_registry is not None:
        modellab_registry.auto_log_experiment(env_globals=globals(), **REGISTRY_INFO)
    else:
        # Cadangan bila utility script belum ditambahkan: cetak payload dengan format IDENTIK
        # supaya scripts/extract_registry.py di GitHub tetap bisa membacanya.
        print("PERINGATAN: modellab_registry tidak ditemukan -> payload dicetak langsung")
        _payload = {**REGISTRY_INFO, "metrics": {"accuracy": float(accuracy), "f1": float(f1_score),
                                                 "precision": float(precision), "recall": float(recall)}}
        print("===REGISTRY_PAYLOAD_START===\n" + json.dumps(_payload) + "\n===REGISTRY_PAYLOAD_END===")
